# Triaxial Compression — Reference State, Profile Evolution & Modulus

Analysis for `triaxial_compression.lmp`. Sections, in the order they run:

| § | what |
|---|---|
| **0** | Combined sweep overview — every `COMP_LEVELS` level overlaid (skipped for a single level) |
| **1** | Reference state (ε = 0): averaged profiles with 95 % CIs |
| **2** | Profile evolution over the production hold (draining transient → plateau) |
| **2b** | $\phi_s$ diagnostic: $W_{s,zz}/V_{\rm solv}$, mass-fraction vs Voronoi — **not** the pore pressure (see §3) |
| **3** | Network (effective) stress & pore pressure (Terzaghi split) |
| **4** | Piston pressure / force histories, and piston–gel contact |
| **5** | Longitudinal modulus $M$: network vs piston estimate |
| **6** | Stress–strain sweep summary |
| **7** | Cooperative diffusivity $D_c$ per strain level |

Set `RUN_ID`, `NSTEPS` and `COMP_LEVELS` in **Config** to match the run, then run
the **sync** cell — it pulls every file the notebook reads (including the
displacement and solvent-virial profiles) in one Expanse login.

Colorblind-friendly palette throughout; gel interior shaded; in/out-gel means annotated.

## Files the notebook reads

The **sync cell** below pulls all of these from Expanse automatically — this
table is the reference for what lands where if you ever copy by hand. Every path
is built from `RUN_ID`, `sim_name` and `COMP_LEVELS` in the Config cell.
Production files carry a `_c<level>` tag; `_ref` files are shared across levels.

**Into** `flow_data_local/compression/<RUN_ID>/`  *(cluster:* `.../output_files/`*)*

| file pattern | cluster subfolder |
|---|---|
| `sigmazz_{polymer,solvent}[_ref]_<sim_name>.dat` | `stress_data/` |
| `virialzz_solvent[_ref]_<sim_name>.dat` — non-volume solvent zz stress, kinetic term included (§2b) | `stress_data/` |
| `strain_zz_<sim_name>.dat`, `strain_piston_<sim_name>.dat` | `stress_data/` |
| `box_dimensions_<sim_name>.dat`, `gel_dimensions_{bb,rg}_<sim_name>.dat`, `polymer_com_<sim_name>.dat` | `stress_data/` |
| `solvent_density_z[_ref]_<sim_name>.dat` | `chemical_potential/` |
| `piston_position_<sim_name>.dat`, `piston_force[_avg]_<sim_name>.dat` | `piston_data/` |
| `disp_z_polymer_<sim_name>.dat` — polymer $u_z(z,t)$ for $D_c$ (§7) | `displacement_data/` |
| `{pairs,polymer_pairs,bonds}[_ref]_<sim_name>.dump` | `pair_data/` |

**Into** `flow_data_local/traj_files.nosync/`  *(cluster:* `.../traj_files/`*)*

| file pattern | cluster subfolder |
|---|---|
| `traj_stress_<sim_name>.lammpstrj`, `traj_ref_<sim_name>.lammpstrj` | `traj_files/` |

Plots are written to `flow_data_local/plots/compression/<RUN_ID>/` (created automatically).

*Not synced:* the run also writes high-resolution `*_fine_support/piston_*` profiles.
This notebook plots the coarse profiles only; add them to `_PROD_DAT` in the sync
cell if you later want high-res panels.

In [ ]:
import numpy as np
import warnings
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from scipy import stats
from scipy.optimize import minimize_scalar
from pathlib import Path

# ---- user rcParams (matches compression_analysis.ipynb) ----
plt.rcParams.update({
    'font.family': 'CMU Serif',
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'CMU Serif',
    'font.size': 20,
    'axes.titlesize': 22,
    'axes.labelsize': 25,
    'xtick.labelsize': 23,
    'ytick.labelsize': 23,
    'legend.fontsize': 23,
    'figure.titlesize': 22,
    'axes.unicode_minus': False,
})

# ---- colorblind-friendly palettes ----
# Wong (2011) categorical palette for the reference single-curve plots.
WONG = {'blue':'#0072B2','orange':'#E69F00','green':'#009E73','vermillion':'#D55E00',
        'skyblue':'#56B4E9','yellow':'#F0E442','reddishpurple':'#CC79A7','black':'#000000'}
# 'cividis' is the most CVD-safe sequential map -> used for the time gradient.
EVO_CMAP = 'cividis'
GEL_SHADE = dict(color='0.6', alpha=0.15, zorder=0)   # neutral grey, CVD-safe

print('Imports + style ready')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG — only change the lines in this block to switch datasets
# ══════════════════════════════════════════════════════════════════════════
# sim_name is the exact suffix LAMMPS appends to every output file:
#     <DATANAME>_<INTERACTION>_<NSTEPS>
DATANAME    = "final_config_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000000"
INTERACTION = "1.0_1.0"                 # epsSS_epsSP for the triaxial run
NSTEPS      = 5000000                   # triaxial production steps.  None -> auto-detect the LARGEST
                                        # NSTEPS present in DATA_DIR (sync first).  Pin it whenever two
                                        # runs of the same DATANAME share a RUN_ID folder.
RUN_ID      = "periodic_rho04_1.4M_5M_twoPlatesMove_3"  # local folder label under flow_data_local/{compression,plots}
                                        # ^ bump this per run so a new NSTEPS gets its own data/plot folder

# ──────────────────────────────────────────────────────────────────────────
#  STRAIN SWEEP LEVELS  ← the one place that defines the sweep
# ──────────────────────────────────────────────────────────────────────────
# These are the APPLIED STRAIN targets (fraction of L0_bb) and MUST match STRAIN_TARGETS=(...) in
# triaxial_compression.batch on the cluster.  triaxial_compression.lmp tags
# EVERY production file with _c<level> — even a single-target run writes its one
# default level (e.g. _c0.10) — so this list is how the notebook finds files.
# There is NO such thing as an un-tagged production run; a single run is just a
# one-element list here.
#   • The sweep-summary plot (§6, last cell) loops over ALL levels below.
#   • The detailed per-level plots (§1–§5) use the ONE level picked by DETAIL_IDX.
#COMP_LEVELS = ["0.05", "0.10", "0.15", "0.20"]   # ← edit to match STRAIN_TARGETS=(...) in the batch
COMP_LEVELS = ["0.10"]
DETAIL_LEVEL = "0.10"   # ← which sweep level the detailed §1–§5 plots below use.
                        #    Must be one of COMP_LEVELS; set None to use the deepest (last).
# ══════════════════════════════════════════════════════════════════════════

assert COMP_LEVELS, "COMP_LEVELS is empty — list at least one pressure level (e.g. [\"0.10\"])."
if DETAIL_LEVEL is None:
    COMP_LEVEL = COMP_LEVELS[-1]
else:
    assert DETAIL_LEVEL in COMP_LEVELS, (
        f"DETAIL_LEVEL {DETAIL_LEVEL!r} is not in COMP_LEVELS {COMP_LEVELS} — "
        "pick one of those strings.")
    COMP_LEVEL = DETAIL_LEVEL
comp_percent = float(COMP_LEVEL)   # applied STRAIN target for the detail level (denominator for M)
print(f'>>> SWEEP has {len(COMP_LEVELS)} strain level(s): {COMP_LEVELS}')
print(f'>>> detailed plots (§1–§5) use strain = {COMP_LEVEL}')

DATA_DIR  = Path("../../flow_data_local/compression") / RUN_ID
PLOT_DIR  = Path("../../flow_data_local/plots/compression") / RUN_ID
TRAJ_DIR  = Path("../../flow_data_local/traj_files.nosync")
# auto-create the local folders for this run (idempotent; safe to re-run)
for _d in (DATA_DIR, PLOT_DIR, TRAJ_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print('folders ready:', DATA_DIR, '|', PLOT_DIR, '|', TRAJ_DIR)

# Build sim_name.  If NSTEPS is None, auto-detect it from the stress files already
# in DATA_DIR (e.g. after the Expanse sync); else use it directly.  Production
# files carry a trailing _c<level> tag and *_ref files do not, so the pattern has
# to allow both -- matching only '..._<digits>.dat' silently misses every
# production file and finds nothing.
if NSTEPS is None:
    import re as _re
    _pat = _re.compile(rf'^sigmazz_polymer(?:_ref)?_{_re.escape(DATANAME)}_'
                       rf'{_re.escape(INTERACTION)}_(\d+)(?:_c[\d.]+)?\.dat$')
    _hits = sorted({int(m.group(1)) for f in DATA_DIR.glob('sigmazz_polymer*.dat')
                    if (m := _pat.match(f.name))})
    if not _hits:
        raise ValueError('NSTEPS is None and no sigmazz_polymer_*.dat in DATA_DIR yet '
                         '-- set NSTEPS explicitly, or run the Expanse sync cell first.')
    NSTEPS = _hits[-1]
    print(f'auto-detected NSTEPS = {NSTEPS} from {len(_hits)} candidate(s): {_hits}')
sim_name = f'{DATANAME}_{INTERACTION}_{NSTEPS}'    # base tag (NO _c); used verbatim by every *_ref file
LEVEL_TAG = f'_c{COMP_LEVEL}'                       # per-level suffix on PRODUCTION files (always set; .lmp always tags _c)

# ---- analysis parameters ----
binWidth      = 2.0      # coarse z-bin (sigma); must match triaxial_compression.lmp
# ---- kinetic (ideal-gas) stress term ------------------------------------
# LAMMPS' `compute stress/atom NULL` does NOT drop the kinetic term.  NULL only
# means "no velocity-bias temperature compute -- use velocities as-is"; and with
# no keywords listed the compute defaults to
#     ke pair bond angle dihedral improper kspace fix
# so EVERY group-based profile triaxial_compression.lmp writes (sigmazz_*,
# sigmaxx/yy_*, stress_{x,y,z}_*, virialzz_solvent, and the fine-grained
# sigmazz_*_fine_*) already carries the kinetic term n(z)*k_B*T.
#
# Verified in the pure-solvent reservoir, where solvent-wall pair coeffs are OFF
# (pair_coeff 3 4 / 3 5 = 0) so every solvent interaction is a 3-3 pair and the
# group value must exceed the ss pair virial by exactly the kinetic term:
#     group W/V_bin = 1.4872    ss pair virial/V_bin = 1.0136
#     difference    = 0.4737    n_s*k_B*T           = 0.4680     (agrees to 1.2%)
#
# The pair/local reconstructions (sigma_s,ss and sigma_p,pp) are built from FORCES
# and so are pure virial -- they need the kinetic term added by hand to be
# comparable to anything else in this notebook.  The traj dumps carry no
# velocities (`id type mol x y z`), so it is added analytically as n(z)*KIN_T from
# the per-bin atom count of the same frame; that is exact on average for an
# equilibrated run thermostatted at KIN_T.  Cell "load ss/pp" prints the reservoir
# residual so the assumption stays checked rather than assumed.
KIN_T         = 1.0      # k_B*T of the run (= temp_target in triaxial_compression.lmp)
ADD_KINETIC   = True     # True  -> ss/pp curves = pair(+bond) virial + n*KIN_T, directly
                         #          comparable to the group profiles (USE THIS)
                         # False -> raw virial-only ss/pp, NOT comparable to the group
                         #          curves; for debugging the reconstruction only
n_curves      = 10       # target time-evolution curves (matches num_stress_curves)
Ncount_min    = 200      # min polymer atoms per z-bin to count it in-gel (D_c mask)
dt_lj         = 0.005    # LJ timestep (triaxial_compression.lmp: timestep_prod); D_c time axis
ci_level      = 0.95
plateau_frac  = 0.5      # fraction of the production HOLD (measured from its end) used
                         # as the EQUILIBRATED piston-pressure plateau for M_piston and
                         # its block-bootstrap CI. 0.5 = last half; more samples => tighter.
gel_thresh    = 0.05     # gel interior = bins where |sigma_p,zz| > gel_thresh*max
flat_tol      = 0.15     # final evolution curve "flat inside gel" if rel. spread < this
wall_margin   = 4.0      # sigma trimmed off support/piston ends before flat test + mean (skip wall-depletion layers)
BOND_SIGN     = +1.0     # flip to -1 if sigma_p,pp comes out opposite the group sigma_p

# ---- file paths ----
def _tag(nm): return '' if nm.endswith('_ref') else LEVEL_TAG   # *_ref files are shared across levels (no _c)
def D(name):  return DATA_DIR / f'{name}_{sim_name}{_tag(name)}.dat'
def Ddump(name): return DATA_DIR / f'{name}_{sim_name}{_tag(name)}.dump'
def T(name):  return TRAJ_DIR / f'{name}_{sim_name}{_tag(name)}.lammpstrj'

# stress (coarse): sigma_zz polymer/solvent, production + reference
F_SZZ_P      = D('sigmazz_polymer');      F_SZZ_S      = D('sigmazz_solvent')
F_SZZ_P_REF  = D('sigmazz_polymer_ref');  F_SZZ_S_REF  = D('sigmazz_solvent_ref')
# NON-VOLUME solvent zz virial W_s,zz (energy units) -> section 2b
F_VIRZZ_S    = D('virialzz_solvent');     F_VIRZZ_S_REF = D('virialzz_solvent_ref')
# solvent density (coarse), production + reference  (cols: ... density/number density/mass)
F_DENS       = DATA_DIR / f'solvent_density_z_{sim_name}{LEVEL_TAG}.dat'
F_DENS_REF   = DATA_DIR / f'solvent_density_z_ref_{sim_name}.dat'
# pair/bond dumps for ss and pp stress
F_PAIRS      = Ddump('pairs');            F_PAIRS_REF      = Ddump('pairs_ref')
F_PPAIRS     = Ddump('polymer_pairs');    F_PPAIRS_REF     = Ddump('polymer_pairs_ref')
F_BONDS      = Ddump('bonds');            F_BONDS_REF      = Ddump('bonds_ref')
F_TRAJSTRESS = T('traj_stress');          F_TRAJREF        = T('traj_ref')
# strain + piston position (to find the compression-halt timestep)
F_STRAIN     = D('strain_zz')
F_STRAIN_PISTON = D('strain_piston')   # step, L0, piston_support_gap, piston_strain (pre-seating .lmp)
F_PISTON_POS = DATA_DIR / f'piston_position_{sim_name}{LEVEL_TAG}.dat'
F_PISTON_FORCE = DATA_DIR / f'piston_force_{sim_name}{LEVEL_TAG}.dat'
F_PISTON_FORCE_AVG = DATA_DIR / f'piston_force_avg_{sim_name}{LEVEL_TAG}.dat'  # LMP block-averaged
# NOTE: per-level disp_z_polymer_*.dat (D_c, §7) paths are built inside load_disp().
print('Config set for', sim_name, '(LEVEL_TAG=' + (LEVEL_TAG or 'none') + ')')

## Sync data from Expanse (only when needed)


In [ ]:
# === Sync triaxial-compression data from Expanse (only when needed) ===
# Equivalent to the volume_of_mixing sync cell, adapted to this notebook's files.
# Pulls exactly what the loader below reads for the current sim_name:
#   • core .dat : sigmazz_(polymer|solvent)[_ref], virialzz_solvent[_ref],
#                 solvent_density_z[_ref], strain_zz, piston_position,
#                 piston_force, disp_z_polymer, box/gel dimensions -> DATA_DIR
#   • pair/bond : pairs[_ref], polymer_pairs[_ref], bonds[_ref] (.dump) -> DATA_DIR
#   • trajs     : traj_stress, traj_ref (.lammpstrj)            -> TRAJ_DIR
# disp_z_polymer (displacement_data/) and virialzz_solvent (stress_data/) are
# pulled HERE — the staging find() walks the whole runs tree by basename, so no
# per-subdir special case is needed and no separate rsync cell.
# An Expanse login happens ONLY if a REQUIRED core file is missing locally.
# Set FORCE_SYNC = True to also refresh the large pair/bond/traj files when the
# core files are already present.
import paramiko, getpass, stat

EXPANSE_HOST = "login.expanse.sdsc.edu"
EXPANSE_USER = "dpollard"
RUNS_ROOT    = "/home/dpollard/Documents/lammps_runs/triaxial_compression"   # working dirs (output_files/*)
TRAJ_ROOT    = "/expanse/lustre/scratch/dpollard/temp_project/lammps_trajectories"  # .lammpstrj on scratch
STAGE_DIR    = f"{RUNS_ROOT}/triaxial_stage"
FORCE_SYNC   = False     # True -> sync even if core files already present (refresh dumps/trajs)

# Files the loader reads (built from the F_* paths defined in the Config cell),
# split by local destination.  REQUIRED = the core files needed for the main plots.
# Files span EVERY level in COMP_LEVELS (the sweep-summary and combined-sweep
# plots need per-level data), plus the shared _ref files pulled once.  Missing
# box_dimensions here is what made the stress-strain sweep plot come up empty.
_PROD_DAT  = ('sigmazz_polymer', 'sigmazz_solvent', 'solvent_density_z',
              'virialzz_solvent',        # non-volume solvent zz virial W_s,zz (section 2b)
              'disp_z_polymer',          # polymer u_z(z,t) for D_c (section 7)
              'strain_zz', 'piston_position', 'piston_force', 'piston_force_avg',
              'box_dimensions',
              # envelope / diagnostic series (Rg vs bounding-box strain, gel band)
              'gel_dimensions_bb', 'gel_dimensions_rg', 'polymer_com',
              'strain_piston')  # boundary (piston-support) strain, new protocol
_PROD_DUMP = ('pairs', 'polymer_pairs', 'bonds')
def _level_data_files(lvl):
    b = f'{sim_name}_c{lvl}'
    return ([DATA_DIR / f'{n}_{b}.dat'  for n in _PROD_DAT]
          + [DATA_DIR / f'{n}_{b}.dump' for n in _PROD_DUMP])
def _level_traj_files(lvl):
    return [TRAJ_DIR / f'traj_stress_{sim_name}_c{lvl}.lammpstrj']

_REF_DATA = [F_SZZ_P_REF, F_SZZ_S_REF, F_VIRZZ_S_REF, F_DENS_REF,
             F_PAIRS_REF, F_PPAIRS_REF, F_BONDS_REF]
DATA_FILES = list(_REF_DATA)
TRAJ_FILES = [F_TRAJREF]
REQUIRED   = [F_SZZ_P_REF, F_SZZ_S_REF, F_DENS_REF]
for _lvl in COMP_LEVELS:
    DATA_FILES += _level_data_files(_lvl)
    TRAJ_FILES += _level_traj_files(_lvl)
    _b = f'{sim_name}_c{_lvl}'
    REQUIRED   += [DATA_DIR / f'sigmazz_polymer_{_b}.dat',
                   DATA_DIR / f'sigmazz_solvent_{_b}.dat',
                   DATA_DIR / f'solvent_density_z_{_b}.dat',
                   DATA_DIR / f'strain_zz_{_b}.dat',
                   DATA_DIR / f'piston_force_{_b}.dat',
                   DATA_DIR / f'box_dimensions_{_b}.dat',
                   DATA_DIR / f'gel_dimensions_bb_{_b}.dat',
                   DATA_DIR / f'disp_z_polymer_{_b}.dat']  # forces login when BB/disp missing

missing_req = [f for f in REQUIRED               if not Path(f).exists()]
missing_all = [f for f in (DATA_FILES+TRAJ_FILES) if not Path(f).exists()]
if not FORCE_SYNC and not missing_req:
    print(f"All {len(REQUIRED)} required files present locally "
          f"({len(missing_all)} optional pair/traj file(s) missing) — skipping Expanse login.")
else:
    why = "FORCE_SYNC" if (FORCE_SYNC and not missing_req) else f"{len(missing_req)} required file(s) missing"
    print(f"Syncing from Expanse ({why}); {len(missing_all)} of "
          f"{len(DATA_FILES)+len(TRAJ_FILES)} target files missing locally.")

    def _bash_list(paths):
        return " ".join(f'"{Path(p).name}"' for p in paths)
    DATA_BN, TRAJ_BN = _bash_list(DATA_FILES), _bash_list(TRAJ_FILES)

    # Stage the newest match for each wanted basename with ONE find per tree
    # (basename->newest index; cp -p preserves mtimes so the SFTP skip fires on
    # reruns).  .dat/.dump come from the runs tree, .lammpstrj from scratch.
    stage_script = r"""
set -u
RUNS="__RUNS__"; TRAJ="__TRAJ__"; STAGE="__STAGE__"
rm -rf "$STAGE"; mkdir -p "$STAGE/data" "$STAGE/traj"
declare -A NEWEST_D
while IFS= read -r line; do p=${line#* }; b=${p##*/}
  [ -z "${NEWEST_D[$b]:-}" ] && NEWEST_D[$b]="$p"
done < <(find "$RUNS" \( -name '*.dat' -o -name '*.dump' \) -not -path '*/triaxial_stage/*' -printf '%T@ %p\n' 2>/dev/null | sort -rn)
for B in __DATA_BN__; do S="${NEWEST_D[$B]:-}"; [ -n "$S" ] && cp -p "$S" "$STAGE/data/" 2>/dev/null || true; done
declare -A NEWEST_T
while IFS= read -r line; do p=${line#* }; b=${p##*/}
  [ -z "${NEWEST_T[$b]:-}" ] && NEWEST_T[$b]="$p"
done < <(find "$TRAJ" "$RUNS" -name '*.lammpstrj' -not -path '*/triaxial_stage/*' -printf '%T@ %p\n' 2>/dev/null | sort -rn)
for B in __TRAJ_BN__; do S="${NEWEST_T[$B]:-}"; [ -n "$S" ] && cp -p "$S" "$STAGE/traj/" 2>/dev/null || true; done
echo "  staged: $(ls "$STAGE/data" 2>/dev/null | wc -l) data, $(ls "$STAGE/traj" 2>/dev/null | wc -l) traj"
"""
    stage_script = (stage_script.replace("__RUNS__", RUNS_ROOT).replace("__TRAJ__", TRAJ_ROOT)
                    .replace("__STAGE__", STAGE_DIR).replace("__DATA_BN__", DATA_BN)
                    .replace("__TRAJ_BN__", TRAJ_BN))

    password = getpass.getpass(f"Expanse password for {EXPANSE_USER}: ")
    totp     = getpass.getpass("TOTP / verification code: ")
    def auth_handler(title, instructions, prompt_list):
        return [password if "password" in p.strip().lower() else totp for p, _ in prompt_list]

    print("Connecting to Expanse...")
    transport = paramiko.Transport((EXPANSE_HOST, 22))
    transport.connect()
    transport.auth_interactive(EXPANSE_USER, auth_handler)
    ssh = paramiko.SSHClient(); ssh._transport = transport

    print("Step 1 — staging files on Expanse...")
    _, stdout, stderr = ssh.exec_command("bash -s", get_pty=False)
    stdout.channel.sendall(stage_script.encode()); stdout.channel.shutdown_write()
    print(stdout.read().decode())

    print("Step 2 — downloading via SFTP (skips files already present)...")
    sftp = ssh.open_sftp()
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    TRAJ_DIR.mkdir(parents=True, exist_ok=True)
    def sftp_pull(remote_dir, local_dir):
        local_dir = Path(local_dir); local_dir.mkdir(parents=True, exist_ok=True)
        try: entries = sftp.listdir_attr(remote_dir)
        except FileNotFoundError: return
        for e in entries:
            rp, lp = f"{remote_dir}/{e.filename}", local_dir / e.filename
            if stat.S_ISDIR(e.st_mode):
                sftp_pull(rp, lp); continue
            if lp.exists() and lp.stat().st_mtime >= e.st_mtime:
                continue
            sftp.get(rp, str(lp))
    sftp_pull(f"{STAGE_DIR}/data", DATA_DIR)
    sftp_pull(f"{STAGE_DIR}/traj", TRAJ_DIR)
    sftp.close(); ssh.close()
    print("Sync complete.")


In [ ]:
# ============================ HELPER FUNCTIONS =============================
def read_print_file(filepath, col_names=None):
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            rows.append([float(v) for v in line.split()])
    arr = np.array(rows)
    if col_names is None: col_names = [f'col_{i}' for i in range(arr.shape[1])]
    return {name: arr[:, i] for i, name in enumerate(col_names)}

def read_ave_time_file(filepath):
    """fix ave/time mode vector -> list of (timestep, bin_idx, values)."""
    out = []
    with open(filepath) as f:
        lines = [l for l in f if not l.startswith('#') and l.strip()]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) == 2:
            ts, nrows = int(parts[0]), int(parts[1])
            vals = []
            for j in range(1, nrows + 1):
                if i + j < len(lines):
                    vp = lines[i + j].split()
                    if len(vp) == 2: vals.append(float(vp[1]))
            if vals: out.append((ts, np.arange(1, len(vals)+1), np.array(vals)))
            i += nrows + 1
        else:
            i += 1
    return out

def read_ave_chunk_file(filepath):
    """fix ave/chunk -> list of (timestep, array[rows, cols]).
    cols: [chunk_id, Coord1, Ncount, val1(, val2...)]."""
    snaps = []
    with open(filepath) as f:
        lines = [l for l in f if l.strip() and not l.startswith('#')]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) in (2, 3):
            try: ts, nch = int(parts[0]), int(parts[1])
            except ValueError:
                i += 1; continue
            rows = []
            for j in range(1, nch + 1):
                if i + j < len(lines): rows.append([float(v) for v in lines[i + j].split()])
            if rows: snaps.append((ts, np.array(rows)))
            i += nch + 1
        else:
            i += 1
    return snaps

def read_strain_file(filepath):
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            rows.append([float(v) for v in line.split()])
    arr = np.array(rows)
    # cols: step  L_initial  L_current  -> eps = (L0 - L)/L0
    ts = arr[:, 0].astype(int); L0 = arr[:, 1]; L = arr[:, 2]
    eps = (L0 - L) / L0
    return ts, eps

def mean_ci(stack, ci=0.95):
    """stack: (n_samples, n_bins) -> (mean, lo, hi) per bin via t-interval.
    All-NaN bins (z-bins outside the data range) return NaN quietly instead of
    emitting a 'Mean of empty slice' RuntimeWarning."""
    stack = np.asarray(stack, float)
    n = stack.shape[0]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        m = np.nanmean(stack, axis=0)
        if n < 2:
            return m, m, m
        se = stats.sem(stack, axis=0, nan_policy='omit')
    half = se * stats.t.ppf(0.5 + ci/2, df=n-1)
    return m, m - half, m + half

def read_pairs_local_dump(filepath):
    """dump local (pair/local) -> list (timestep, box, data[n,7])
    data cols: id1 id2 type1 type2 fx fy fz."""
    frames = []
    with open(filepath) as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        if lines[i].strip() == 'ITEM: TIMESTEP':
            ts = int(lines[i+1]); n = int(lines[i+3])
            xb = list(map(float, lines[i+5].split())); yb = list(map(float, lines[i+6].split()))
            zb = list(map(float, lines[i+7].split()))
            box = {'x': tuple(xb), 'y': tuple(yb), 'z': tuple(zb)}
            s = i + 9
            rows = [[float(v) for v in lines[s+j].split()] for j in range(n) if s+j < len(lines)]
            frames.append((ts, box, np.array(rows) if rows else np.zeros((0, 7))))
            i = s + n
        else:
            i += 1
    return frames

def read_bonds_dump(filepath):
    """dump local (bond/local) -> list (timestep, box, data[n,5])
    data cols: batom1 batom2 btype force dist."""
    frames = []
    with open(filepath) as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        if lines[i].strip() == 'ITEM: TIMESTEP':
            ts = int(lines[i+1]); n = int(lines[i+3])
            xb = list(map(float, lines[i+5].split())); yb = list(map(float, lines[i+6].split()))
            zb = list(map(float, lines[i+7].split()))
            box = {'x': tuple(xb), 'y': tuple(yb), 'z': tuple(zb)}
            s = i + 9
            rows = [[float(v) for v in lines[s+j].split()] for j in range(n) if s+j < len(lines)]
            frames.append((ts, box, np.array(rows) if rows else np.zeros((0, 5))))
            i = s + n
        else:
            i += 1
    return frames

def read_box(dumpfile):
    """Box bounds + edge lengths from the first frame of any LAMMPS dump.
    The single box-header parser for the whole notebook (dumps share the header
    layout, and the box is fixed here — only the piston moves inside it)."""
    with open(dumpfile) as f:
        head = [next(f) for _ in range(9)]
    b = {k: tuple(map(float, head[5 + i].split())) for i, k in enumerate('xyz')}
    b.update(lx=b['x'][1]-b['x'][0], ly=b['y'][1]-b['y'][0], lz=b['z'][1]-b['z'][0])
    return b

def rolling_mean(y, win):
    """Centered moving average; the window shrinks at the edges (min_periods=1).
    Used to tame the raw piston force/pressure, which is thermally noisy because
    the piston is undamped by design — we time-average rather than damp."""
    y = np.asarray(y, float); n = len(y)
    if win <= 1 or n == 0: return y.copy()
    half = win // 2; csum = np.concatenate(([0.0], np.cumsum(y))); out = np.empty(n)
    for k in range(n):
        lo, hi = max(0, k - half), min(n, k + half + 1)
        out[k] = (csum[hi] - csum[lo]) / (hi - lo)
    return out

def _stream_traj_positions(traj_file, target_ts, types_keep):
    """Return {ts: (box, {id:(x,y,z)})} for atoms whose type is in types_keep."""
    out = {}
    with open(traj_file) as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        if 'ITEM: TIMESTEP' in lines[i]:
            ts = int(lines[i+1]); n = int(lines[i+3])
            if ts in target_ts:
                xb = list(map(float, lines[i+5].split())); yb = list(map(float, lines[i+6].split()))
                zb = list(map(float, lines[i+7].split()))
                box = {'x': tuple(xb), 'y': tuple(yb), 'z': tuple(zb)}
                pos = {}
                for j in range(n):
                    p = lines[i+9+j].split()
                    if int(p[1]) in types_keep:
                        pos[int(p[0])] = (float(p[3]), float(p[4]), float(p[5]))
                out[ts] = (box, pos)
            i += 9 + n
        else:
            i += 1
    return out

def stream_traj_frames(traj_file, want_ts):
    """{ts: (box, types[N], xyz[N,3])} for want_ts — ALL atom types, streamed so a
    100 MB dump is never held in memory.  Companion to _stream_traj_positions()
    above, which instead filters by type and keys positions by atom id."""
    out, want = {}, {int(t) for t in want_ts}
    with open(traj_file) as f:
        while True:
            line = f.readline()
            if not line: break
            if not line.startswith('ITEM: TIMESTEP'): continue
            ts = int(f.readline()); f.readline(); n = int(f.readline()); f.readline()
            box = {k: tuple(map(float, f.readline().split())) for k in 'xyz'}
            cols = f.readline().split()[2:]
            if ts not in want:
                for _ in range(n): f.readline()
                continue
            ti, xi, yi, zi = (cols.index(c) for c in ('type', 'x', 'y', 'z'))
            a = np.empty((n, 4))
            for j in range(n):
                p = f.readline().split()
                a[j] = (float(p[ti]), float(p[xi]), float(p[yi]), float(p[zi]))
            out[ts] = (box, a[:, 0].astype(int), a[:, 1:])
    return out

print('Readers defined')

In [ ]:
# ===== solvent-only (ss) and polymer-only (pp) stress from pair/bond dumps =====
# These are reconstructed from FORCES (compute pair/local, compute bond/local) and
# are therefore PURE VIRIAL.  The group-based profiles LAMMPS writes via
# `compute stress/atom NULL` are NOT virial-only -- NULL suppresses only the
# velocity-bias temperature compute, and with no keywords the compute defaults to
# `ke pair bond angle dihedral improper kspace fix`.  So comparing the two requires
# adding the kinetic term here; ADD_KINETIC / KIN_T in the Config cell control it.
#
# What each object contains, per bin volume:
#   group solvent (sigmazz_solvent_*.dat) = W_ss/V + n_s*kT + (1/2)*W_sp/V
#   group polymer (sigmazz_polymer_*.dat) = W_pp/V + n_p*kT + (1/2)*W_sp/V
#   this cell's sigma_s,ss (ADD_KINETIC)  = W_ss/V + n_s*kT
#   this cell's sigma_p,pp (ADD_KINETIC)  = W_pp/V + n_p*kT
# so after adding kinetic the leftover (group - reconstructed) is (1/2)*W_sp/V, the
# solvent-polymer CROSS virial -- the same number for both groups, since stress/atom
# hands each atom half of every pair virial it takes part in.  That is a free
# consistency check, and it must go to ~0 in the pure-solvent reservoir.
#
# Binning caveat, unchanged by any of this: the virial here is binned by PAIR
# MIDPOINT while LAMMPS' stress/atom bins by ATOM POSITION (each atom carrying half
# of its pair virials).  The two agree wherever the profile is flat over a pair
# cutoff and differ only in the ~1 sigma-wide interfacial layers.  The kinetic term
# is per-atom and so is binned by atom position, matching LAMMPS exactly.
def _virial_zz_from_pairs(pair_data, pos, box, binWidth, type_filter):
    """sigma_zz(z) from pair/local forces, both atoms in type_filter (a set).
    Returns (z_bins, sigma_zz) -- VIRIAL ONLY.  The kinetic term is added separately
    by _kinetic_zz_profile() inside compute_ss_stress()/compute_pp_stress()."""
    lx = box['x'][1]-box['x'][0]; ly = box['y'][1]-box['y'][0]; lz = box['z'][1]-box['z'][0]
    zlo = box['z'][0]
    nb = int(round(lz / binWidth)); binvol = lx*ly*binWidth
    z_bins = zlo + (np.arange(nb)+0.5)*binWidth
    szz = np.zeros(nb)
    if len(pair_data):
        m = np.isin(pair_data[:,2].astype(int), list(type_filter)) & \
            np.isin(pair_data[:,3].astype(int), list(type_filter))
        sp = pair_data[m]
        if len(sp):
            id1 = sp[:,0].astype(int); id2 = sp[:,1].astype(int); fz = sp[:,6]
            keep = np.array([(a in pos and b in pos) for a,b in zip(id1,id2)])
            if keep.any():
                id1=id1[keep]; id2=id2[keep]; fz=fz[keep]
                p1 = np.array([pos[a] for a in id1]); p2 = np.array([pos[b] for b in id2])
                dz = p2[:,2]-p1[:,2]; dz -= lz*np.round(dz/lz)
                w = -(dz*fz)
                zmid = p1[:,2] + dz*0.5
                bi = ((zmid - zlo)/binWidth).astype(int)
                ok = (bi>=0)&(bi<nb)
                np.add.at(szz, bi[ok], w[ok])
    return z_bins, szz/binvol

def _bond_virial_zz(bond_data, pos, box, binWidth, bond_sign=1.0):
    """FENE bond sigma_zz(z): W_zz = bond_sign*(force/|r|)*dz^2, midpoint-binned."""
    lx = box['x'][1]-box['x'][0]; ly = box['y'][1]-box['y'][0]; lz = box['z'][1]-box['z'][0]
    zlo = box['z'][0]
    nb = int(round(lz / binWidth)); binvol = lx*ly*binWidth
    szz = np.zeros(nb)
    if len(bond_data):
        b1 = bond_data[:,0].astype(int); b2 = bond_data[:,1].astype(int); force = bond_data[:,3]
        keep = np.array([(a in pos and b in pos) for a,b in zip(b1,b2)])
        if keep.any():
            b1=b1[keep]; b2=b2[keep]; force=force[keep]
            p1=np.array([pos[a] for a in b1]); p2=np.array([pos[b] for b in b2])
            dr = p2-p1
            dr[:,0]-=lx*np.round(dr[:,0]/lx); dr[:,1]-=ly*np.round(dr[:,1]/ly); dr[:,2]-=lz*np.round(dr[:,2]/lz)
            r = np.sqrt((dr**2).sum(1)); r[r==0]=np.nan
            w = bond_sign*(force/r)*dr[:,2]**2
            zmid = p1[:,2] + dr[:,2]*0.5
            bi = ((zmid - zlo)/binWidth).astype(int); ok=(bi>=0)&(bi<nb)&np.isfinite(w)
            np.add.at(szz, bi[ok], w[ok])
    return szz/binvol

def _kinetic_zz_profile(pos, box, binWidth, kT):
    """Ideal-gas (kinetic) zz stress per z-bin for the atoms in `pos` ({id:(x,y,z)}).

        sigma_kin,zz(z) = N(z) * kT / V_bin

    Binned by ATOM position -- exactly how LAMMPS' compute stress/atom assigns its
    kinetic contribution -- on the frame's own grid, matching _virial_zz_from_pairs.
    Uses equipartition (<m v_z^2> = kT) because the traj dumps carry no velocities;
    exact on average for an equilibrated run thermostatted at kT."""
    lx = box['x'][1]-box['x'][0]; ly = box['y'][1]-box['y'][0]; lz = box['z'][1]-box['z'][0]
    zlo = box['z'][0]
    nb = int(round(lz / binWidth)); binvol = lx*ly*binWidth
    z_bins = zlo + (np.arange(nb)+0.5)*binWidth
    if not pos:
        return z_bins, np.zeros(nb)
    zc = np.fromiter((p[2] for p in pos.values()), float, len(pos))
    bi = ((zc - zlo)/binWidth).astype(int)
    ok = (bi >= 0) & (bi < nb)
    return z_bins, np.bincount(bi[ok], minlength=nb)[:nb] * kT / binvol

def compute_ss_stress(pairs_file, traj_file, binWidth, z_target,
                      add_kinetic=None, kT=None):
    """Per-frame sigma_s,zz^ss(z) on z_target: ss PAIR virial + solvent kinetic term.

    add_kinetic / kT default to ADD_KINETIC / KIN_T from the Config cell.  With
    add_kinetic=True the result is directly comparable to the group-based
    sigmazz_solvent_*.dat (which already includes kinetic), and the remaining gap
    between them is (1/2)*W_sp -- the solvent-polymer cross virial.
    Returns (timesteps, stack[n,nz])."""
    add_kinetic = ADD_KINETIC if add_kinetic is None else add_kinetic
    kT = KIN_T if kT is None else kT
    pf = read_pairs_local_dump(pairs_file)
    if not pf: return [], np.zeros((0,len(z_target)))
    tgt = {ts for ts,_,_ in pf}
    traj = _stream_traj_positions(traj_file, tgt, {3})
    ts_out, stack = [], []
    for ts, box, pdata in pf:
        if ts not in traj: continue
        _, pos = traj[ts]
        zb, szz = _virial_zz_from_pairs(pdata, pos, box, binWidth, {3})
        if add_kinetic:
            szz = szz + _kinetic_zz_profile(pos, box, binWidth, kT)[1]
        stack.append(np.interp(z_target, zb, szz, left=np.nan, right=np.nan)); ts_out.append(ts)
    return ts_out, np.array(stack)

def compute_pp_stress(ppairs_file, bonds_file, traj_file, binWidth, z_target,
                      bond_sign=1.0, add_kinetic=None, kT=None):
    """Per-frame sigma_p,zz^pp(z) on z_target: pp pair virial + FENE bond virial +
    polymer kinetic term.  add_kinetic / kT default to ADD_KINETIC / KIN_T (Config
    cell); with add_kinetic=True this is comparable to sigmazz_polymer_*.dat and the
    remaining gap is (1/2)*W_sp, the same cross virial the solvent side sees."""
    add_kinetic = ADD_KINETIC if add_kinetic is None else add_kinetic
    kT = KIN_T if kT is None else kT
    pf = read_pairs_local_dump(ppairs_file)
    bf = {ts:(box,data) for ts,box,data in read_bonds_dump(bonds_file)} if Path(bonds_file).exists() else {}
    if not pf: return [], np.zeros((0,len(z_target)))
    tgt = {ts for ts,_,_ in pf}
    traj = _stream_traj_positions(traj_file, tgt, {1,2})
    ts_out, stack = [], []
    for ts, box, pdata in pf:
        if ts not in traj: continue
        _, pos = traj[ts]
        zb, szz = _virial_zz_from_pairs(pdata, pos, box, binWidth, {1,2})
        if ts in bf:
            szz = szz + _bond_virial_zz(bf[ts][1], pos, box, binWidth, bond_sign)
        if add_kinetic:
            szz = szz + _kinetic_zz_profile(pos, box, binWidth, kT)[1]
        stack.append(np.interp(z_target, zb, szz, left=np.nan, right=np.nan)); ts_out.append(ts)
    return ts_out, np.array(stack)

print(f'ss / pp stress reconstruction defined  (ADD_KINETIC={ADD_KINETIC}, KIN_T={KIN_T})')

## Load data: coordinates, gel bounds, reference & production profiles

In [ ]:
# ---- box geometry (fixed: only the piston moves) + wall positions ----
# One read_box() call feeds every geometric constant the notebook uses, so the
# cross-section and bin volume can never drift out of sync between sections.
BOX0       = read_box(F_PAIRS_REF)
Z_LO, Z_HI = BOX0['z']
LX, LY, LZ = BOX0['lx'], BOX0['ly'], BOX0['lz']
AREA_XY    = LX * LY              # fixed cross-section (no lateral barostat in Phase 2)
V_BIN      = AREA_XY * binWidth   # coarse z-bin volume; matches the .lmp chunk volume
zn = lambda z: (np.asarray(z, float) - Z_LO) / LZ      # fractional box height, ~[0, 1]

def _wall_z_first_frame(traj, types=(4, 5)):
    """Mean z of each atom type in the FIRST frame of a LAMMPS dump trajectory."""
    zc = {t: [] for t in types}
    if not Path(traj).exists():
        return {t: np.nan for t in types}
    with open(traj) as f:
        cols = None
        for line in f:                                  # advance to the ATOMS header
            if line.startswith('ITEM: ATOMS'):
                cols = line.split()[2:]; break
        ti, zi = cols.index('type'), cols.index('z')
        for line in f:
            if line.startswith('ITEM:'): break          # stop at next frame
            p = line.split(); t = int(float(p[ti]))
            if t in zc: zc[t].append(float(p[zi]))
    return {t: (np.mean(v) if v else np.nan) for t, v in zc.items()}
_wz = _wall_z_first_frame(F_TRAJREF, (4, 5))
z_support = _wz.get(4, np.nan)                          # type 4 = frozen support sheet
z_piston  = _wz.get(5, np.nan)                          # type 5 = piston sheet
print(f'box (fixed): Lx={LX:.2f} Ly={LY:.2f} Lz={LZ:.2f}  |  z in [{Z_LO:.2f}, {Z_HI:.2f}]  |  '
      f'A={AREA_XY:.2f}  V_bin={V_BIN:.2f}')
print(f'support (type4) z = {z_support:.2f}  |  piston (type5) z = {z_piston:.2f}')

# piston z(t): output only during compression; np.interp clamps to the held
# post-halt value, so any evolution timestep maps to the compressed position.
if Path(F_PISTON_POS).exists():
    _pp = np.loadtxt(F_PISTON_POS, comments='#')
    _pt, _pz = _pp[:, 0], _pp[:, 1]
    def piston_z_at(t): return float(np.interp(t, _pt, _pz))
else:
    def piston_z_at(t): return z_piston
print(f'piston z(t): {_pt[0]:.1f} -> {_pt[-1]:.1f} steps, '
      f'z {_pz[0]:.1f} -> {_pz[-1]:.1f}' if Path(F_PISTON_POS).exists() else 'piston z(t): file missing')

# ---- coarse sigma_zz (production) -> z grid + total/partial time series ----
szz_p = read_ave_time_file(F_SZZ_P)
szz_s = read_ave_time_file(F_SZZ_S)
n_prod = len(szz_p)
prod_ts   = np.array([szz_p[i][0] for i in range(n_prod)])
sig_p_zz  = [szz_p[i][2] for i in range(n_prod)]
sig_s_zz  = [szz_s[i][2] for i in range(n_prod)]
sig_t_zz  = [sig_p_zz[i] + sig_s_zz[i] for i in range(n_prod)]      # total sigma_zz
bins_z    = szz_p[0][1]
z_coords  = Z_LO + bins_z * binWidth - binWidth/2.0   # bin centers in real box coords
print(f'coarse stress: {n_prod} production snapshots, {len(z_coords)} z-bins '
      f'[{z_coords.min():.1f}, {z_coords.max():.1f}]')

# ---- reference coarse sigma_zz (multi-snapshot -> mean + CI) ----
szz_p_ref = read_ave_time_file(F_SZZ_P_REF)
szz_s_ref = read_ave_time_file(F_SZZ_S_REF)
ref_p_stack = np.array([s[2] for s in szz_p_ref])
ref_s_stack = np.array([s[2] for s in szz_s_ref])
ref_t_stack = ref_p_stack + ref_s_stack
print(f'reference stress: {len(szz_p_ref)} snapshots')

# total sigma_zz reference mean/CI
sig_t_ref_m, sig_t_ref_lo, sig_t_ref_hi = mean_ci(ref_t_stack, ci_level)
# group-based polymer reference (for the pp sign-validation cell)
sig_p_ref_m = np.nanmean(ref_p_stack, axis=0)

# ---- gel bounds from reference polymer sigma_zz ----
_pm = np.abs(sig_p_ref_m); _pmax = float(_pm.max())
_gel = (_pm > gel_thresh*_pmax) if _pmax > 0 else np.zeros(len(z_coords), bool)
z_gel_lo = float(z_coords[_gel].min()) if _gel.any() else z_coords[0]
z_gel_hi = float(z_coords[_gel].max()) if _gel.any() else z_coords[-1]
in_gel  = (z_coords >= z_gel_lo) & (z_coords <= z_gel_hi)
print(f'gel interior: z in [{z_gel_lo:.1f}, {z_gel_hi:.1f}]  ({in_gel.sum()} bins)')

# ---- measurement (plateau) window: strain-controlled HOLD ----
# The piston is FROZEN at the prescribed strain while the network stress relaxes;
# the measured Rg strain sits near comp_percent.  The measurement window is the
# last plateau_frac of the production HOLD snapshots (prod_ts).  halt_ts (name
# kept for downstream post_halt()) marks the START of that equilibrated window.
strain_ts, strain_eps = read_strain_file(F_STRAIN)
_t0p, _t1p = float(prod_ts[0]), float(prod_ts[-1])
halt_ts = int(_t1p - plateau_frac * (_t1p - _t0p))     # start of the equilibrated plateau
# evol_from: START of the production HOLD (= the compression halt).  Used ONLY by
# the profile-evolution PLOT so it shows the full draining transient -> plateau.
# The MEASUREMENT window (M_piston, pore pressure, BB/boundary strain) stays on
# halt_ts, and D_c fits its own disp window -- neither depends on evol_from.
evol_from = int(_t0p)
_plat = strain_ts >= halt_ts
eps_measured = float(np.mean(strain_eps[_plat])) if _plat.sum() >= 1 else float(strain_eps[-1])
print(f'measurement window: steps >= {halt_ts} (last {plateau_frac:.0%} of production)')
print(f'measured Rg strain (plateau mean) = {eps_measured:.4f}  |  end Rg strain = {float(strain_eps[-1]):.4f}')

# ---- APPLIED (prescribed) strain + measured BB strain (verification) ----
# STRAIN-CONTROLLED: the piston is driven to displacement = strain*L0_bb, so the
# APPLIED strain is comp_percent (the detail level) -- this is the M denominator.
# The measured BB thickness change should come out ~= comp_percent, confirming
# the plates bracket the gel (the bounding-box method).
eps_applied = comp_percent
_bb = None
try:
    _bb = np.atleast_2d(np.loadtxt(DATA_DIR / f'gel_dimensions_bb_{sim_name}{LEVEL_TAG}.dat', comments='#'))
except Exception:
    pass
if _bb is not None and _bb.shape[1] >= 4 and _bb[0, 3] != 0:
    _L0bb = _bb[0, 3]
    _platbb = _bb[:, 0] >= halt_ts
    eps_bb_measured = float(np.mean((_L0bb - _bb[_platbb, 3]) / _L0bb)) if _platbb.sum() else float((_L0bb - _bb[-1, 3]) / _L0bb)
else:
    eps_bb_measured = float('nan')
print(f'APPLIED strain (denominator) = {eps_applied:.4f}  |  measured BB strain (plateau) = {eps_bb_measured:.4f}')

# ---- BOUNDARY (piston-support) strain: DIAGNOSTIC ONLY ----
# strain_piston cols: step, L0, piston_support_gap, piston_strain.  With the
# body-force seating removed, this OVER-reads: L0 includes the solvent void the
# piston closes while translating the (uncompressed) gel onto the support, so it
# is NOT used for M -- the measured Rg strain above is.  Kept for the diagnostic.
if Path(F_STRAIN_PISTON).exists():
    _psd = np.atleast_2d(np.loadtxt(F_STRAIN_PISTON, comments='#'))
    piston_strain_ts, piston_strain_series = _psd[:, 0], _psd[:, 3]
    L0_boundary  = float(_psd[0, 1])
    eps_boundary = float(np.median(piston_strain_series[piston_strain_ts >= halt_ts]))
    HAS_BOUNDARY_STRAIN = True
    print(f'boundary strain (piston-support, DIAGNOSTIC) = {eps_boundary:.4f}  [over-reads: includes void]')
else:
    piston_strain_ts = piston_strain_series = None; L0_boundary = None; eps_boundary = None
    HAS_BOUNDARY_STRAIN = False
    print('NOTE: no strain_piston file.')


In [ ]:
# ---- solvent density (production + reference): cols chunk,Coord1,Ncount,n_dens,m_dens ----
def _load_density(path):
    snaps = read_ave_chunk_file(path)
    ts = np.array([s[0] for s in snaps])
    z  = snaps[0][1][:, 1]
    nden = np.array([s[1][:, 3] for s in snaps])        # density/number
    mden = np.array([s[1][:, 4] for s in snaps])        # density/mass
    return ts, z, nden, mden

dens_ts, dens_z, dens_n, dens_m = _load_density(F_DENS)
rdens_ts, rdens_z, rdens_n, rdens_m = _load_density(F_DENS_REF)
print(f'density: {len(dens_ts)} production, {len(rdens_ts)} reference snapshots on {len(dens_z)} bins')

# reference mass density mean/CI
rho_ref_m, rho_ref_lo, rho_ref_hi = mean_ci(rdens_m, ci_level)
# bulk reference solvent mass density rho_{s,0} = mean over reservoir bins (high-density)
_rmax = float(np.nanmax(rho_ref_m)); _res = rho_ref_m >= 0.85*_rmax
rho_s0 = float(np.nanmean(rho_ref_m[_res]))
print(f'rho_s,0 (bulk reservoir reference mass density) = {rho_s0:.4f}')

# reference mass fraction stack = each ref snapshot / rho_s0
mf_ref_stack = rdens_m / rho_s0
mf_ref_m, mf_ref_lo, mf_ref_hi = mean_ci(mf_ref_stack, ci_level)

In [ ]:
# ---- solvent-only (ss) and polymer-only (pp) stress: reference + production ----
# Reference: multi-frame *_ref dumps -> mean + CI.  Production: filter post-halt.
ss_ref_m = ss_ref_lo = ss_ref_hi = None
ss_prod_ts = None; ss_prod_stack = None
if F_PAIRS_REF.exists() and F_TRAJREF.exists():
    _ts, _stk = compute_ss_stress(F_PAIRS_REF, F_TRAJREF, binWidth, z_coords)
    if len(_stk): ss_ref_m, ss_ref_lo, ss_ref_hi = mean_ci(_stk, ci_level)
    print(f'sigma_s,ss reference: {len(_ts)} frames')
else:
    print('NOTE: ss reference dumps missing -', F_PAIRS_REF.name, '/', F_TRAJREF.name)

if F_PAIRS.exists() and F_TRAJSTRESS.exists():
    ss_prod_ts, ss_prod_stack = compute_ss_stress(F_PAIRS, F_TRAJSTRESS, binWidth, z_coords)
    ss_prod_ts = np.array(ss_prod_ts)
    print(f'sigma_s,ss production: {len(ss_prod_ts)} frames')
else:
    print('NOTE: ss production dumps missing')

pp_ref_m = pp_ref_lo = pp_ref_hi = None
pp_prod_ts = None; pp_prod_stack = None
if F_PPAIRS_REF.exists() and F_TRAJREF.exists():
    _ts, _stk = compute_pp_stress(F_PPAIRS_REF, F_BONDS_REF, F_TRAJREF, binWidth, z_coords, BOND_SIGN)
    if len(_stk): pp_ref_m, pp_ref_lo, pp_ref_hi = mean_ci(_stk, ci_level)
    print(f'sigma_p,pp reference: {len(_ts)} frames')
if F_PPAIRS.exists() and F_TRAJSTRESS.exists():
    pp_prod_ts, pp_prod_stack = compute_pp_stress(F_PPAIRS, F_BONDS, F_TRAJSTRESS, binWidth, z_coords, BOND_SIGN)
    pp_prod_ts = np.array(pp_prod_ts)
    print(f'sigma_p,pp production: {len(pp_prod_ts)} frames')

# ---- kinetic / cross-virial consistency check ----------------------------
# (group - reconstructed) must equal (1/2)*W_sp/V once the kinetic term is added:
#   * ~0 in the pure-solvent reservoir  <- this is the test of ADD_KINETIC / KIN_T,
#     because there is no polymer there to produce a cross term at all
#   * the SAME positive number on the solvent and polymer sides inside the gel,
#     since compute stress/atom hands each atom half of every pair virial
# A nonzero reservoir residual means KIN_T is wrong or the run was not at that
# temperature; a solvent/polymer mismatch inside the gel means one reconstruction
# is incomplete (missing bond term, wrong BOND_SIGN, unmatched traj frames).
def _plateau(ts, stack):
    """Bin-wise mean over the equilibrated plateau (ts >= halt_ts)."""
    ts = np.asarray(ts); stack = np.asarray(stack)
    m = ts >= halt_ts
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        return np.nanmean(stack[m], axis=0) if m.any() else stack[-1]

if (ss_prod_stack is not None and len(ss_prod_stack)
        and pp_prod_stack is not None and len(pp_prod_stack)):
    _gs = _plateau(prod_ts, np.asarray(sig_s_zz))       # group solvent
    _gp = _plateau(prod_ts, np.asarray(sig_p_zz))       # group polymer
    _rs = _plateau(ss_prod_ts, ss_prod_stack)           # ss pair (+kinetic)
    _rp = _plateau(pp_prod_ts, pp_prod_stack)           # pp pair+bond (+kinetic)
    _zf = (z_coords - z_coords.min()) / (z_coords.max() - z_coords.min())
    _res = (_zf >= 0.90) & (_zf <= 0.99)
    # Trim wall_margin off BOTH ends of the gel: pair_coeff 1 4 / 1 5 / 2 4 / 2 5
    # are all 1.0, so the group polymer stress picks up half the polymer-WALL cross
    # virial in the contact layers while the pp reconstruction (polymer group
    # pair/local, so pp pairs only) does not.  Comparing there is not apples-to-apples.
    _core = (in_gel & (z_coords >= z_gel_lo + wall_margin)
                    & (z_coords <= piston_z_at(prod_ts[-1]) - wall_margin))
    _xs = _gs - _rs                                    # solvent side: (1/2)W_sp/V
    _xp = _gp - _rp                                    # polymer side: same quantity
    print(f'\ncross-virial check (ADD_KINETIC={ADD_KINETIC}, KIN_T={KIN_T}, '
          f'plateau mean over ts >= {halt_ts}):')
    print(f'  reservoir residual, solvent side = {np.nanmean(_xs[_res]):+.4f}'
          f'   <- should be ~0 (no polymer there)')
    print(f'  gel-core (1/2)W_sp/V, solvent    = {np.nanmean(_xs[_core]):+.4f}')
    print(f'  gel-core (1/2)W_sp/V, polymer    = {np.nanmean(_xp[_core]):+.4f}'
          f'   <- should match the line above')
    if not ADD_KINETIC:
        print('  NOTE: ADD_KINETIC=False -- these residuals still contain n*kT '
              'and the ss/pp plots are NOT comparable to the group curves.')

In [ ]:
# ---- sigma_p,pp sign validation (group sigma_p_ref should track pp+kin) ----
# The group-based reference polymer stress (incl. bonds + 1/2 ps cross) should
# have the SAME sign and comparable magnitude as the pp reconstruction.  If the
# pp curve is mirrored about zero, set BOND_SIGN = -1 in the Config cell.
if pp_ref_m is not None:
    _ig = in_gel
    corr = np.corrcoef(pp_ref_m[_ig], sig_p_ref_m[_ig])[0, 1]
    print(f'corr(sigma_p,pp_ref, group sigma_p_ref) inside gel = {corr:+.3f}')
    if corr < 0:
        print('  >>> NEGATIVE correlation: sigma_p,pp likely sign-flipped. '
              'Set BOND_SIGN = -1.0 in Config and re-run.')
    else:
        print('  sign looks consistent (positive correlation).')
else:
    print('pp reference unavailable - skipping sign check.')

## Plotting helpers

In [ ]:
def _fmt_val_unc(v, u):
    """value ± uncertainty, uncertainty rounded to 2 sig figs (value matched)."""
    v = float(v)
    if np.isfinite(u) and u > 0:
        dec = int(np.clip(1 - np.floor(np.log10(u)), 0, 6))
        return f'{v:.{dec}f} \u00b1 {u:.{dec}f}'
    return f'{v:.3g}'

def _fmt_mu(vals):
    """mean ± std (2 sig figs) over finite values."""
    a = np.asarray(vals, float); a = a[np.isfinite(a)]
    if a.size == 0: return 'n/a'
    return _fmt_val_unc(np.mean(a), np.std(a))

def _means_text(z, curve, in_gel):
    """Mean ± scatter of `curve` inside and outside the gel as formatted text."""
    mi = np.nanmean(curve[in_gel]) if in_gel.any() else np.nan
    mo = np.nanmean(curve[~in_gel]) if (~in_gel).any() else np.nan
    return mi, mo, f'mean in gel = {_fmt_mu(curve[in_gel])}\nmean out gel = {_fmt_mu(curve[~in_gel])}'

def _final_flat_inside(curve, z, in_gel, tol):
    """Flat = small linear TREND across the gel (noise-robust, unlike raw std).
    Returns True when |slope|*interior_width / |mean| < tol."""
    v = curve[in_gel]; zz = np.asarray(z)[in_gel]
    m = np.isfinite(v); v, zz = v[m], zz[m]
    if len(v) < 3: return False
    slope = np.polyfit(zz, v, 1)[0]
    rise = abs(slope) * (zz.max() - zz.min())
    denom = max(abs(np.mean(v)), 1e-9)
    return (rise / denom) < tol

def shade_gel(ax):
    ax.axvspan(zn(z_gel_lo), zn(z_gel_hi), **GEL_SHADE)

def mark_walls(ax, ts=None):
    """Frozen support (type4, solid) + piston (type5, dash-dot).  Lines only,
    no text labels.  ts=None -> reference (uncompressed) piston position;
    ts given -> piston per evolution timestep at its COMPRESSED position, with
    the final one dark and earlier ones faded."""
    if np.isfinite(z_support):
        ax.axvline(zn(z_support), color=WONG['black'], ls='-', lw=1.5, alpha=0.85, zorder=4)
    if ts is None:
        pistons = [(z_piston, True)]
    else:
        ts = np.asarray(ts)
        pistons = [(piston_z_at(t), i == len(ts) - 1) for i, t in enumerate(ts)]
    for pz, is_last in pistons:
        if np.isfinite(pz):
            ax.axvline(zn(pz), color=('0.15' if is_last else '0.7'), ls='-.',
                       lw=(1.6 if is_last else 1.0), alpha=(0.9 if is_last else 0.35),
                       zorder=(4 if is_last else 3))

def plot_reference(ax, z, m, lo, hi, color, ylabel, title, annotate=True):
    zx = zn(z)
    ax.fill_between(zx, lo, hi, color=color, alpha=0.25, lw=0, zorder=2)
    ax.plot(zx, m, '-', color=color, lw=2.5, zorder=3)
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
    shade_gel(ax); mark_walls(ax)
    ax.set_xlabel(r'$z/L_z$'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xlim(0, 1); ax.grid(alpha=0.3)
    if annotate:
        mi, mo, txt = _means_text(z, m, in_gel)
        ax.text(0.02, 0.03, txt, transform=ax.transAxes, va='bottom', ha='left',
                fontsize=15, bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))

def subsample(ts, stack, k):
    """Evenly pick up to k snapshots (keep order; always include the last)."""
    n = len(ts)
    if n <= k: idx = np.arange(n)
    else:      idx = np.unique(np.linspace(0, n-1, k).round().astype(int))
    return ts[idx], stack[idx]

def plot_evolution(ax, z, ts, stack, ylabel, title):
    """Time-coloured profiles (cividis); final curve bold black; gel shaded.
    Mean-in/out annotation ONLY if the final curve is flat inside the gel."""
    ts = np.asarray(ts); stack = np.asarray(stack)
    norm = Normalize(vmin=ts.min(), vmax=ts.max())
    cmap = plt.get_cmap(EVO_CMAP)
    zx = zn(z)
    for i in range(len(ts)):
        is_last = (i == len(ts)-1)
        ax.plot(zx, stack[i], '-',
                color=('k' if is_last else cmap(norm(ts[i]))),
                lw=(3.5 if is_last else 1.6),
                alpha=(1.0 if is_last else 0.75),
                zorder=(5 if is_last else 3))
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
    shade_gel(ax); mark_walls(ax, ts)
    ax.set_xlabel(r'$z/L_z$'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xlim(0, 1); ax.grid(alpha=0.3)
    sm = plt.cm.ScalarMappable(cmap=EVO_CMAP, norm=norm); sm.set_array([])
    cb = ax.figure.colorbar(sm, ax=ax, fraction=0.046, pad=0.02); cb.set_label('timestep')
    # gel is squeezed between the support and the CURRENT piston: restrict the
    # interior to that window so the reservoir past the piston never enters the
    # flatness test or the mean (that jump is what made noisy panels read "not flat").
    z_pist = piston_z_at(np.asarray(ts)[-1])
    _zz = np.asarray(z)
    interior = in_gel & (_zz >= z_gel_lo + wall_margin) & (_zz <= z_pist - wall_margin)
    if _final_flat_inside(stack[-1], z, interior, flat_tol):
        ax.text(0.02, 0.03, 'final (equilibrated)\nmean in gel = ' + _fmt_mu(stack[-1][interior]),
                transform=ax.transAxes, va='bottom', ha='left', fontsize=14,
                bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))
    else:
        ax.text(0.02, 0.03, 'final not flat inside\n(means omitted)', transform=ax.transAxes,
                va='bottom', ha='left', fontsize=13, color='0.35')

def robust_ylim(ax, curves, zmask=None, pad=0.12, qlo=2, qhi=98, include_zero=True):
    """Frame the y-axis to the bulk of the data (qlo–qhi percentiles), ignoring
    a few extreme wall-edge spike bins.  curves: iterable of 1-D arrays (each a
    profile); zmask: optional boolean mask selecting which bins to consider."""
    vals = []
    for c in curves:
        c = np.asarray(c, float)
        if zmask is not None: c = c[zmask]
        c = c[np.isfinite(c)]
        if c.size: vals.append(c)
    if not vals:
        return
    v = np.concatenate(vals)
    lo, hi = np.percentile(v, [qlo, qhi])
    if include_zero:
        lo, hi = min(lo, 0.0), max(hi, 0.0)
    if hi <= lo: hi = lo + 1.0
    d = (hi - lo) * pad
    ax.set_ylim(lo - d, hi + d)

def _autocorr_time(x):
    """Integrated autocorrelation time (samples), Sokal automatic windowing."""
    x = np.asarray(x, float); n = len(x)
    if n < 4: return 1.0
    x = x - x.mean(); var = np.dot(x, x) / n
    if var <= 0: return 1.0
    fx = np.fft.rfft(x, n=2*n)
    acf = np.fft.irfft(fx * np.conj(fx))[:n].real / (var * n)
    tau = 1.0
    for W in range(1, n):
        tau = 1.0 + 2.0 * np.sum(acf[1:W+1])
        if W >= 5.0 * tau: break
    return float(max(tau, 1.0))

def block_bootstrap_ci(x, ci=0.95, n_boot=2000, block=None, seed=12345):
    """Circular block-bootstrap CI for the MEAN of an autocorrelated series.
    Returns (mean, lo, hi, block_len, tau).  Resamples contiguous wrap-around
    blocks of length ~2*tau so within-block correlation is preserved and the CI
    reflects the effective sample count ~N/block, not N."""
    x = np.asarray(x, float); n = len(x)
    if n < 2:
        m = float(x.mean()) if n else np.nan
        return m, np.nan, np.nan, 1, 1.0
    tau = _autocorr_time(x)
    if block is None:
        block = int(np.clip(np.ceil(2.0 * tau), 1, max(1, n // 2)))
    rng = np.random.default_rng(seed)
    nblk = int(np.ceil(n / block))
    starts = rng.integers(0, n, size=(n_boot, nblk))
    idx = (starts[:, :, None] + np.arange(block)[None, None, :]) % n
    idx = idx.reshape(n_boot, -1)[:, :n]
    means = x[idx].mean(axis=1)
    lo, hi = np.percentile(means, [100*(1-ci)/2, 100*(1+ci)/2])
    return float(x.mean()), float(lo), float(hi), int(block), float(tau)

print('Plot helpers ready')

## 0 — Combined sweep overview (all strain levels, shown first)

These panels overlay **every** `COMP_LEVELS` level on shared axes so the whole
stress–strain sweep is visible at a glance, before the detailed single-level
sections below.  They are produced **only when `COMP_LEVELS` has more than one
level** (`IS_SWEEP`); for a single run this section is skipped and you drop
straight into the per-level plots.

Colour = strain level (see legend).  Within each level the relaxation evolution
is drawn faint→bold (early holds transparent, the final equilibrated profile
thick and opaque).  Requires the per-level files to be synced first (run the
sync cell above — it now pulls every level).

In [ ]:
# ==========================================================================
#  0 — COMBINED SWEEP: load every level once, into SWEEP (list of dicts)
# ==========================================================================
# Self-contained: reuses the readers/helpers defined above but builds its own
# per-level paths and its own z-grid (the box is fixed, only the piston moves,
# so every level shares Z_LO/LZ).  ss stress and piston data are optional per
# level and degrade gracefully if a file is missing.

IS_SWEEP = len(COMP_LEVELS) > 1

# categorical, CVD-safe colour per level (Wong palette, cycled if >8 levels)
LEVEL_COLORS = [WONG[k] for k in ('blue','vermillion','green','reddishpurple',
                                  'orange','skyblue','yellow','black')]
def level_color(i): return LEVEL_COLORS[i % len(LEVEL_COLORS)]

def _net_from_total(sig_p_list, sig_s_list, zc):
    """Terzaghi split per level (mirrors section 3): network = total - pore
    baseline (flat reservoir at z/Lz~0.95); membrane from final polymer stress."""
    tot = np.asarray([sig_p_list[i] + sig_s_list[i] for i in range(len(sig_p_list))])
    zf = (zc - zc.min()) / (zc.max() - zc.min())
    bw = (zf >= 0.91) & (zf <= 0.99) & (zf < 0.995)
    spf = np.abs(sig_p_list[-1]); thr = gel_thresh * float(np.nanmax(spf))
    mem = spf > thr
    zlo = float(zc[mem].min()) if mem.any() else zc[0]
    zhi = float(zc[mem].max()) if mem.any() else zc[-1]
    in_mem = (zc >= zlo) & (zc <= zhi)
    net = np.zeros_like(tot); pore = np.zeros(len(tot)); pore_h = np.zeros(len(tot))
    for i in range(len(tot)):
        v = tot[i][bw]; v = v[np.isfinite(v)]
        p0 = float(np.nanmean(v)) if len(v) else 0.0
        se = float(stats.sem(v)) if len(v) > 1 else 0.0
        tcr = stats.t.ppf(0.5 + ci_level/2, df=max(len(v)-1, 1))
        pore[i] = p0; pore_h[i] = tcr * se; net[i] = tot[i] - p0
    return tot, net, pore, pore_h, in_mem, (zlo, zhi)

def load_level(lvl):
    b = f'{sim_name}_c{lvl}'
    P  = lambda n, e='dat': DATA_DIR / f'{n}_{b}.{e}'
    Tf = lambda n: TRAJ_DIR / f'{n}_{b}.lammpstrj'
    fp, fs = P('sigmazz_polymer'), P('sigmazz_solvent')
    if not (fp.exists() and fs.exists()):
        print(f'  level {lvl}: sigmazz files missing — skipping'); return None
    sp, ss = read_ave_time_file(fp), read_ave_time_file(fs)
    if not sp or not ss:
        print(f'  level {lvl}: empty sigmazz files — skipping'); return None
    ts = np.array([x[0] for x in sp])
    sig_p = [x[2] for x in sp]; sig_s = [x[2] for x in ss]
    zc = Z_LO + sp[0][1] * binWidth - binWidth/2.0
    tot, net, pore, pore_h, in_mem, memb = _net_from_total(sig_p, sig_s, zc)
    L = dict(lvl=lvl, ts=ts, z=zc, tot=tot, net=net, pore=pore, pore_h=pore_h,
             in_mem=in_mem, memb=memb)
    # solvent density + mass fraction
    fd = P('solvent_density_z')
    if fd.exists():
        d_ts, d_z, d_n, d_m = _load_density(fd)
        # Normalise mass fraction by THIS level's OWN bulk reservoir density (the
        # coexisting bath, z/Lz in [0.90, 0.99] of the final snapshot) so every
        # reservoir plateau lands at 1.0.  A single global rho_s0 leaves each
        # level slightly off 1 because the reservoir density itself drifts as the
        # gel consolidates and the reservoir grows.
        _zf = (d_z - d_z.min()) / (d_z.max() - d_z.min())
        _res = (_zf >= 0.90) & (_zf <= 0.99)
        rho_res = float(np.nanmean(d_m[-1][_res])) if _res.any() else rho_s0
        L.update(d_ts=d_ts, d_z=d_z, d_m=d_m, d_mf=d_m / rho_res, rho_res=rho_res)
    # solvent-only ss stress (needs pair dump + stress traj)
    fpairs, ftr = P('pairs','dump'), Tf('traj_stress')
    if fpairs.exists() and ftr.exists():
        try:
            _t, _stk = compute_ss_stress(fpairs, ftr, binWidth, zc)
            if len(_stk): L['ss_ts'], L['ss'] = np.array(_t), _stk
        except Exception as e:
            print(f'  level {lvl}: ss stress failed ({e})')
    # strain — the M DENOMINATOR is the PRESCRIBED APPLIED strain (float(lvl)),
    # consistent with sections 5 and 6.  The run is strain-controlled: the piston
    # is driven to displacement = strain*L0_bb, so float(lvl) IS the applied
    # (bounding-box-referenced) strain.  The measured Rg / BB strains are kept as
    # DIAGNOSTICS ONLY (previously L['eps'] used the Rg plateau, which under-reads
    # the true strain -- and under-reads most at small strain -- so M=stress/eps_Rg
    # was spuriously inflated at low strain, faking a decrease of M with strain).
    L['eps'] = float(lvl)          # applied (prescribed) strain = M denominator
    L['eps_kind'] = 'applied'
    fstr = P('strain_zz')
    if fstr.exists():
        s_ts, s_eps = read_strain_file(fstr)
        _t0e, _t1e = float(ts[0]), float(ts[-1])
        _wlo = _t1e - plateau_frac * (_t1e - _t0e)
        _pm = s_ts >= _wlo
        L['eps_rg'] = float(np.mean(s_eps[_pm])) if _pm.sum() >= 1 else float(s_eps[-1])  # DIAGNOSTIC
        # measured BB strain (should ~= applied) — DIAGNOSTIC
        fbb = P('gel_dimensions_bb')
        if Path(fbb).exists():
            _b = np.atleast_2d(np.loadtxt(fbb, comments='#'))
            if _b.shape[1] >= 4 and _b[0, 3] != 0:
                _pmb = _b[:, 0] >= _wlo
                L['eps_bb'] = float(np.mean((_b[0, 3] - _b[_pmb, 3]) / _b[0, 3])) if _pmb.sum() else float((_b[0, 3] - _b[-1, 3]) / _b[0, 3])
        # boundary strain kept for diagnostics only (over-reads: includes void)
        fps = P('strain_piston')
        if Path(fps).exists():
            _p = np.atleast_2d(np.loadtxt(fps, comments='#'))
            L['eps_boundary'] = float(np.median(_p[_p[:, 0] >= _wlo, 3])); L['L0'] = float(_p[0, 1])
        L['eps_ts'] = s_ts; L['eps_series'] = s_eps  # Rg strain series (for diagnostics)
    # cross-section area (lx*ly): from box_dimensions tail, else the ref box header
    fb = P('box_dimensions'); area = None
    if fb.exists():
        B = np.loadtxt(fb, comments='#')
        if B.ndim == 2 and B.shape[1] >= 3: area = float(np.mean(B[-5:,1]*B[-5:,2]))
    if area is None:
        area = AREA_XY          # fixed cross-section from the Config/load cell
    L['area'] = area
    # piston force / pressure
    ff = P('piston_force')
    if ff.exists():
        pf = read_print_file(ff, col_names=['step','Fz'])
        L['pf_step'], L['pf_F'] = pf['step'].astype(int), pf['Fz']
        if area: L['pf_P'] = pf['Fz'] / area
    ffa = P('piston_force_avg')
    if ffa.exists() and area:
        pfa = read_print_file(ffa, col_names=['step','Fz'])
        L['pfa_step'], L['pfa_P'] = pfa['step'].astype(int), pfa['Fz'] / area
    # longitudinal modulus, two estimates
    if L.get('eps', 0) > 0:
        mn = net[-1][in_mem] / L['eps']; mn = mn[np.isfinite(mn)]
        if len(mn):
            L['M_net'], L['M_net_lo'], L['M_net_hi'] = mean_ci(mn, ci_level)
        if area and ('pf_P' in L or 'pfa_P' in L):
            # Plateau window = final coarse-stress epoch of the hold.  Feed the RAW
            # piston pressure to a circular BLOCK BOOTSTRAP (accounts for the
            # autocorrelation of the undamped piston directly); fall back to the
            # block-averaged curve, then last points, if raw is unavailable.
            if 'pf_P' in L:
                _st, _P, _src = np.asarray(L['pf_step']),  np.asarray(L['pf_P']),  'raw'
            else:
                _st, _P, _src = np.asarray(L['pfa_step']), np.asarray(L['pfa_P']), 'block-avg'
            # Equilibrated plateau = last `plateau_frac` of the production HOLD
            # (piston pressure has relaxed by then); more samples -> tighter CI.
            _t0, _t1 = float(L['ts'][0]), float(L['ts'][-1])
            _win_lo = _t1 - plateau_frac * (_t1 - _t0)
            _sel = _st >= _win_lo
            Pwin = _P[_sel]
            if len(Pwin) < 2: Pwin = _P[-10:]
            Pf, Plo, Phi, _blk, _tau = block_bootstrap_ci(Pwin, ci=ci_level)
            L['M_pist']    = Pf / L['eps']
            L['M_pist_lo'] = Plo / L['eps']; L['M_pist_hi'] = Phi / L['eps']
            L['M_pist_err']= 0.5 * (L['M_pist_hi'] - L['M_pist_lo'])
            L['P_final'] = Pf; L['P_src'] = _src
            L['M_pist_block'] = _blk; L['M_pist_tau'] = _tau
    return L

if IS_SWEEP:
    print(f'Loading {len(COMP_LEVELS)} sweep levels: {COMP_LEVELS}')
    SWEEP = [L for L in (load_level(l) for l in COMP_LEVELS) if L is not None]
    SWEEP.sort(key=lambda L: float(L['lvl']))
    print(f'>>> loaded {len(SWEEP)} level(s): {[L["lvl"] for L in SWEEP]}')
else:
    SWEEP = []
    print('Single run (one level in COMP_LEVELS) — combined sweep plots below are skipped.')

In [ ]:
# ==========================================================================
#  0 — COMBINED SWEEP: profile-evolution overlays (all levels on shared axes)
# ==========================================================================
def overlay_evolution(ax, zkey, tskey, stackkey, ylabel, title,
                      autoscale_mask=None):
    """Overlay each level's relaxation evolution.  Colour = level; within a
    level alpha ramps early->late and the final curve is bold."""
    all_curves = []
    for i, L in enumerate(SWEEP):
        if stackkey not in L or L[stackkey] is None or not len(L[stackkey]):
            continue
        ts, stack = subsample(np.asarray(L[tskey]), np.asarray(L[stackkey]), n_curves)
        zx = zn(L[zkey]); col = level_color(i); nc = len(ts)
        for j in range(nc):
            last = (j == nc - 1)
            ramp = 0.20 + 0.80 * (j / max(nc - 1, 1))
            ax.plot(zx, stack[j], '-', color=col,
                    lw=(3.0 if last else 1.1),
                    alpha=(1.0 if last else 0.55 * ramp),
                    zorder=(5 if last else 3))
            if last: all_curves.append(stack[j])
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.4)
    ax.set_xlim(0, 1); ax.grid(alpha=0.3)
    ax.set_xlabel(r'$z/L_z$'); ax.set_ylabel(ylabel); ax.set_title(title)
    if all_curves:
        robust_ylim(ax, all_curves, zmask=autoscale_mask, pad=0.15)

if IS_SWEEP and SWEEP:
    handles = [Line2D([0],[0], color=level_color(i), lw=3,
                      label=fr'$P={float(L["lvl"]):.2f}$')
               for i, L in enumerate(SWEEP)]

    fig, ax = plt.subplots(2, 3, figsize=(22, 12), constrained_layout=True)
    fig.suptitle(f'Combined sweep — profile evolution, all levels:  {sim_name}',
                 fontsize=15, fontweight='bold')

    # membrane-interior + reservoir mask (box-fixed) for network-stress autoscale
    _z0 = SWEEP[0]['z']
    _mem_lo = min(L['memb'][0] for L in SWEEP)
    _net_mask = (_z0 >= _mem_lo + wall_margin) & (_z0 <= _z0.max())

    overlay_evolution(ax[0,0], 'z','ts','tot',
                      r'$\sigma_{zz}^{t}(z,t)$', r'(a) Total stress $\sigma_{zz}^{t}$')
    overlay_evolution(ax[0,1], 'z','ss_ts','ss',
                      r'$\sigma_{s,zz}^{ss}(z,t)$', r'(b) Solvent-only stress $\sigma_{s,zz}^{ss}$')
    overlay_evolution(ax[0,2], 'd_z','d_ts','d_m',
                      r'$\rho_s(z,t)\ (m\,\sigma^{-3})$', r'(c) Solvent mass density $\rho_s$')
    overlay_evolution(ax[1,0], 'd_z','d_ts','d_mf',
                      r'$\rho_s/\rho_{s,0}$', r'(d) Solvent mass fraction $\rho_s/\rho_{s,0}$')
    ax[1,0].axhline(1.0, color='k', ls=':', lw=1.2, alpha=0.6)
    overlay_evolution(ax[1,1], 'z','ts','net',
                      r"$\sigma'_{zz}(z,t)$", r"(e) Network stress $\sigma'_{zz}$",
                      autoscale_mask=_net_mask)

    # (f) pore pressure: flat baseline per level -> final value vs strain target
    axf = ax[1,2]
    for i, L in enumerate(SWEEP):
        eps = L.get('eps', float(L['lvl']))
        axf.errorbar([eps], [L['pore'][-1]], yerr=[L['pore_h'][-1]], fmt='o', ms=11,
                     color=level_color(i), capsize=6, lw=2)
        # label with the applied pressure P (x-value is the measured strain).
        axf.annotate(fr'P={L["lvl"]}', (eps, L['pore'][-1]),
                     textcoords='offset points', xytext=(7, 4), fontsize=10,
                     color=level_color(i))
    axf.set_xlabel(r'gel strain  $\varepsilon$'); axf.set_ylabel(r'$p_{\mathrm{pore}}$ (final)')
    axf.set_title(r'(f) Pore pressure (equilibrated) vs strain'); axf.grid(alpha=0.3)

    for a in (ax[0,0], ax[0,1], ax[0,2], ax[1,0], ax[1,1]):
        a.legend(handles=handles, fontsize=11, loc='best', title='applied P')

    out = PLOT_DIR / f'sweep_profiles_combined_{sim_name}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
else:
    print('combined profile overlays skipped (not a sweep / no levels loaded)')

In [ ]:
# ==========================================================================
#  0 — COMBINED SWEEP: piston pressure & force histories (all levels)
# ==========================================================================
if IS_SWEEP and SWEEP:
    fig, (axP, axF) = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)
    fig.suptitle(f'Combined sweep — piston pressure & force histories:  {sim_name}',
                 fontsize=15, fontweight='bold')
    handles = []
    for i, L in enumerate(SWEEP):
        col = level_color(i)
        lab = fr'$\varepsilon_\mathrm{{target}}={float(L["lvl"]):.2f}$'
        if 'pf_P' in L:
            axP.plot(L['pf_step'], L['pf_P'], '-', color=col, lw=0.8, alpha=0.20)
            axP.plot(L['pf_step'], rolling_mean(L['pf_P'], 21), '-', color=col, lw=2.4, alpha=0.95)
        if 'pf_F' in L:
            axF.plot(L['pf_step'], L['pf_F'], '-', color=col, lw=0.8, alpha=0.20)
            axF.plot(L['pf_step'], rolling_mean(L['pf_F'], 21), '-', color=col, lw=2.4, alpha=0.95)
        handles.append(Line2D([0],[0], color=col, lw=3, label=lab))
    axP.axhline(0, color='k', ls='--', lw=0.8, alpha=0.4)
    axP.set_xlabel('step'); axP.set_ylabel(r'$P = F_z/A$  (LJ / $\sigma^2$)')
    axP.set_title('(a) piston pressure vs step'); axP.grid(alpha=0.3)
    axF.axhline(0, color='k', ls='--', lw=0.8, alpha=0.4)
    axF.set_xlabel('step'); axF.set_ylabel(r'$F_z$  (LJ)')
    axF.set_title('(b) piston force vs step'); axF.grid(alpha=0.3)
    axP.legend(handles=handles, fontsize=11, loc='best', title='sweep level')
    out = PLOT_DIR / f'sweep_piston_history_combined_{sim_name}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
else:
    print('combined piston histories skipped (not a sweep / no levels loaded)')

### Uncertainty on the piston modulus: circular block bootstrap

The piston is undamped, so its force series is thermally noisy **and
autocorrelated** — the naïve standard error $\sigma/\sqrt{N}$ (which assumes
independent samples) understates the uncertainty on the plateau-mean pressure.
We instead use a **circular block bootstrap**. For a plateau series
$P_1,\dots,P_N$ we estimate the integrated autocorrelation time $\tau$ and set a
block length $\ell\approx2\tau$; we then draw $\lceil N/\ell\rceil$ contiguous
wrap-around blocks at random, concatenate them to a length-$N$ surrogate,
recompute the mean, and repeat $B=2000$ times. Resampling whole blocks preserves
within-block correlation, so the spread of the bootstrap means reflects the
*effective* sample count $\sim N/\ell$ rather than $N$. The 95% CI is the
2.5/97.5 percentiles of the bootstrap means, propagated to
$M_\mathrm{piston}=P/\varepsilon$. The network modulus keeps its 95% CI from the
bin-to-bin spatial scatter of $\sigma'_{zz}$. This applies to both the combined
sweep panel below and the single-level §5 comparison.

In [ ]:
# ==========================================================================
#  0 — COMBINED SWEEP: longitudinal modulus M (both estimates) vs strain
# ==========================================================================
# M_network = <sigma'_zz>_membrane / eps   and   M_piston = (F_z/A) / eps,
# one point per level.  Agreement across strains validates the Terzaghi split;
# the initial secant slope of P vs eps is the small-deformation modulus.
if IS_SWEEP and SWEEP:
    eps = np.array([L.get('eps', float(L['lvl'])) for L in SWEEP])
    have_net  = [('M_net'  in L) for L in SWEEP]
    have_pist = [('M_pist' in L) for L in SWEEP]

    fig, (axM, axPP) = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
    fig.suptitle(f'Combined sweep — longitudinal modulus & stress–strain:  {sim_name}',
                 fontsize=15, fontweight='bold')

    # (a) the two M estimates vs strain
    en = eps[have_net]
    if any(have_net):
        Mn  = np.array([L['M_net']    for L in SWEEP if 'M_net' in L])
        Mnl = np.array([L['M_net_lo'] for L in SWEEP if 'M_net' in L])
        Mnh = np.array([L['M_net_hi'] for L in SWEEP if 'M_net' in L])
        axM.errorbar(en, Mn, yerr=[Mn-Mnl, Mnh-Mn], fmt='o-', ms=10, lw=2,
                     color=WONG['blue'], capsize=6,
                     label=r"network  $\langle\sigma'_{zz}\rangle_\mathrm{mem}/\varepsilon$")
    if any(have_pist):
        ep = eps[have_pist]
        Mp  = np.array([L['M_pist']    for L in SWEEP if 'M_pist' in L])
        Mpl = np.array([L['M_pist_lo'] for L in SWEEP if 'M_pist' in L])
        Mph = np.array([L['M_pist_hi'] for L in SWEEP if 'M_pist' in L])
        axM.errorbar(ep, Mp, yerr=[Mp-Mpl, Mph-Mp], fmt='s-', ms=10, lw=2,
                     color=WONG['vermillion'], capsize=6,
                     label=r'piston  $(F_z/A)/\varepsilon$  (block-bootstrap 95% CI)')
    axM.set_xlabel(r'gel strain  $\varepsilon$'); axM.set_ylabel(r'$M$  (LJ units)')
    axM.set_title('(a) longitudinal modulus, two estimates'); axM.grid(alpha=0.3)
    axM.legend(fontsize=13, loc='best')

    # (b) stress-strain: piston pressure vs strain (secant modulus = slope)
    if any(have_pist):
        ep = eps[have_pist]
        Pp = np.array([L['P_final'] for L in SWEEP if 'M_pist' in L])
        order = np.argsort(ep); ep, Pp = ep[order], Pp[order]
        axPP.plot(ep, Pp, 'o-', lw=2, color=WONG['green'])
        for i,(e,p) in enumerate(zip(ep,Pp)):
            axPP.annotate(f'{SWEEP[i]["lvl"]}', (e,p), textcoords='offset points',
                          xytext=(6,6), fontsize=12)
        if len(ep) >= 2:
            M_init = (Pp[1]-Pp[0])/(ep[1]-ep[0]); M_sec = (Pp[-1]-Pp[0])/(ep[-1]-ep[0])
            axPP.set_title(fr'(b) $P$ vs $\varepsilon$   $M_\mathrm{{init}}\approx{M_init:.3g}$,  '
                           fr'$M_\mathrm{{secant}}\approx{M_sec:.3g}$')
        else:
            axPP.set_title(r'(b) piston pressure vs strain')
    axPP.set_xlabel(r'gel strain  $\varepsilon$')
    axPP.set_ylabel(r'$P_{piston}=\langle F_z\rangle/A$  (LJ)')
    axPP.grid(alpha=0.3)

    out = PLOT_DIR / f'sweep_modulus_combined_{sim_name}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
else:
    print('combined modulus plot skipped (not a sweep / no levels loaded)')

## Strain diagnostic — translation vs. compression  (sweep-independent)

A check on *what the strain is actually doing* under the applied displacement. The
gel starts floating just above the support; the piston is driven down, pushing
the slab through the solvent and onto the support, then compresses it. Two
things change the gel's thickness — genuine **compression** and the slab
**translating** onto the support — and only the first is elastic strain.

The modulus $M$ does **not** use either curve below as its denominator: $M$ is
divided by the **prescribed applied strain** (`comp_percent`), which is defined
as a fraction of the seated bounding-box thickness $L_0^{BB}$ (the piston is
driven to displacement $=\varepsilon\,L_0^{BB}$). The Rg and BB curves here are
**consistency checks** — the measured BB strain should come out $\approx$ the
applied strain, confirming the plates bracket the gel.

- **(a)** strain vs step from two fully-swollen-referenced thickness measures:
  bounding-box $\varepsilon_{BB}$ (the strain measure consistent with the $M$
  denominator) and Rg-based $\varepsilon_{Rg}$ (robust body-thickness check,
  immune to translation). The faint thick line is the piston–support boundary
  strain $\varepsilon_\mathrm{piston}$, which OVER-reads (its $L_0$ includes the
  solvent void) — diagnostic only.
- **(b)** z-positions vs step: piston (solid), gel top/bottom (dashed band), and
  the support (black). The band first **translates** downward (settling), then
  **thins** (compression under load).

Uses the fine `strain_freq` recording in `triaxial_compression.lmp`.

In [ ]:
# ==========================================================================
#  STRAIN DIAGNOSTIC  (runs for a single level or a full sweep)
# ==========================================================================
def _load2c(path):
    try:
        a = np.loadtxt(path, comments='#'); return np.atleast_2d(a)
    except Exception:
        return None

_levels = COMP_LEVELS if COMP_LEVELS else [COMP_LEVEL]
fig, (axA, axB) = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)
fig.suptitle(f'Strain diagnostic (translation vs. compression):  {sim_name}',
             fontsize=14, fontweight='bold')
_any = False
for i, lvl in enumerate(_levels):
    b  = f'{sim_name}_c{lvl}'
    S  = _load2c(DATA_DIR / f'strain_zz_{b}.dat')          # step, L0_rg, Lz_rg
    BB = _load2c(DATA_DIR / f'gel_dimensions_bb_{b}.dat')  # step, lx, ly, lz
    PP = _load2c(DATA_DIR / f'piston_position_{b}.dat')    # step, z
    CM = _load2c(DATA_DIR / f'polymer_com_{b}.dat')        # step, cx, cy, cz
    RG = _load2c(DATA_DIR / f'gel_dimensions_rg_{b}.dat')  # step, lx, ly, lz
    if S is None or S.shape[1] < 3:
        continue
    _any = True; col = level_color(i)
    st, L0, Lrg = S[:, 0], S[:, 1], S[:, 2]
    eps_rg = (L0 - Lrg) / L0
    axA.plot(st, eps_rg, '-', color=col, lw=2.2, label=fr'$\varepsilon_{{Rg}}$ (target={lvl})')
    if BB is not None and BB.shape[1] >= 4 and BB[0, 3] != 0:
        axA.plot(BB[:, 0], (BB[0, 3] - BB[:, 3]) / BB[0, 3], '--', color=col, lw=1.5, alpha=0.7)
    PS = _load2c(DATA_DIR / f'strain_piston_{b}.dat')  # step, L0, gap, piston_strain
    if PS is not None and PS.shape[1] >= 4:
        axA.plot(PS[:, 0], PS[:, 3], '-', color=col, lw=3.2, alpha=0.35,
                 label=fr'$\varepsilon_\mathrm{{piston}}$ (target={lvl}, diag)')
    # shade the measurement plateau window (last plateau_frac of the run)
    _wlo = st[-1] - plateau_frac * (st[-1] - st[0])
    axA.axvspan(_wlo, st[-1], color=col, alpha=0.06)
    _pm = st >= _wlo
    _epl = float(np.mean(eps_rg[_pm])) if _pm.sum() else float(eps_rg[-1])
    axA.annotate(f'plateau ε={_epl:.3f}', (st[-1], eps_rg[-1]),
                 textcoords='offset points', xytext=(-6, 0), ha='right',
                 va='center', fontsize=9, color=col)
    if PP is not None and PP.shape[1] >= 2:
        axB.plot(PP[:, 0], PP[:, 1], '-', color=col, lw=2.0, label=fr'piston (target={lvl})')
    if CM is not None and RG is not None and CM.shape[1] >= 4 and RG.shape[1] >= 4:
        lz_cm = np.interp(CM[:, 0], RG[:, 0], RG[:, 3])
        top, bot = CM[:, 3] + 0.5*lz_cm, CM[:, 3] - 0.5*lz_cm
        axB.plot(CM[:, 0], top, '--', color=col, lw=1.1, alpha=0.7)
        axB.plot(CM[:, 0], bot, '--', color=col, lw=1.1, alpha=0.7)
        axB.fill_between(CM[:, 0], bot, top, color=col, alpha=0.06)
    # BB gel edges vs plate faces -- the reference-uncompressed / overlap check:
    # gel_top_bb should sit on the piston face and gel_bot_bb on the support face.
    EG = _load2c(DATA_DIR / f'gel_edges_{b}.dat')  # step, gel_bot_bb, gel_top_bb, piston_z, support_z
    if EG is not None and EG.shape[1] >= 5:
        axB.plot(EG[:, 0], EG[:, 2], '-', color=col, lw=2.4, label=fr'gel top/bot BB (target={lvl})')
        axB.plot(EG[:, 0], EG[:, 1], '-', color=col, lw=2.4)
        axB.plot(EG[:, 0], EG[:, 3], ':',  color='k',   lw=1.8, alpha=0.85)   # piston face
        axB.plot(EG[:, 0], EG[:, 4], '-.', color='0.4', lw=1.8, alpha=0.85)   # support face
if _any and np.isfinite(z_support):
    axB.axhline(z_support, color='k', ls='-', lw=1.6, alpha=0.85, label='support (type 4)')
axA.set_xlabel('step'); axA.set_ylabel(r'compression strain  $\varepsilon=(L_0-L)/L_0$')
axA.set_title(r'(a) strain: solid $\varepsilon_{Rg}$, dashed $\varepsilon_{BB}$, shaded plateau', fontsize=15)
axA.grid(alpha=0.3); axA.legend(fontsize=10, loc='best')
axB.set_xlabel('step'); axB.set_ylabel(r'$z$  ($\sigma$)')
axB.set_title('(b) z-positions: solid=gel BB top/bot, dotted=piston, dash-dot=support\n(BB edges should overlap the plate faces => plates bracket the gel)', fontsize=12)
axB.grid(alpha=0.3); axB.legend(fontsize=10, loc='best')
if _any:
    out = PLOT_DIR / f'strain_diagnostic_{sim_name}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
else:
    plt.close(fig); print('strain diagnostic skipped (no strain_zz files found for COMP_LEVELS)')

## 1 — Reference-state profiles (ε = 0) with confidence intervals

Averaged over the 150k pre-compression window (piston held at v = 0).  Bands are
the 95 % CIs across the reference snapshots; grey shading marks the gel interior.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
fig.suptitle(f'Reference state (ε = 0):  {sim_name}', fontsize=14, fontweight='bold')

# (a) total sigma_zz = sigma_p,zz + sigma_s,zz
plot_reference(axes[0,0], z_coords, sig_t_ref_m, sig_t_ref_lo, sig_t_ref_hi,
               WONG['blue'], r'$\sigma_{zz}^{t}(z)$', r'(a) Total stress $\sigma_{zz}^{t}=\sigma_{p,zz}+\sigma_{s,zz}$')

# (b) solvent-only sigma_s,ss
if ss_ref_m is not None:
    plot_reference(axes[0,1], z_coords, ss_ref_m, ss_ref_lo, ss_ref_hi,
                   WONG['vermillion'], r'$\sigma_{s,zz}^{ss}(z)$', r'(b) Solvent-only stress $\sigma_{s,zz}^{ss}$ (ss pair + kinetic)')
else:
    axes[0,1].text(0.5,0.5,'ss reference\nunavailable',ha='center',va='center',transform=axes[0,1].transAxes)
    axes[0,1].set_title(r'(b) Solvent-only stress $\sigma_{s,zz}^{ss}$ (ss pair + kinetic)')

# (c) solvent mass density rho_s(z)
plot_reference(axes[1,0], dens_z, rho_ref_m, rho_ref_lo, rho_ref_hi,
               WONG['green'], r'$\rho_s(z)\ (m\,\sigma^{-3})$', r'(c) Solvent mass density $\rho_s$')
axes[1,0].axhline(rho_s0, color=WONG['black'], ls=':', lw=1.5, alpha=0.7)
axes[1,0].text(0.98, 0.05, r'$\rho_{s,0}$'+f' = {rho_s0:.3f}', transform=axes[1,0].transAxes,
               ha='right', va='bottom', fontsize=15)

# (d) mass fraction rho_s/rho_s0
plot_reference(axes[1,1], dens_z, mf_ref_m, mf_ref_lo, mf_ref_hi,
               WONG['reddishpurple'], r'$\rho_s/\rho_{s,0}$', r'(d) Solvent mass fraction $\rho_s/\rho_{s,0}$')
axes[1,1].axhline(1.0, color='k', ls=':', lw=1.2, alpha=0.6)

out = PLOT_DIR / f'reference_state_{sim_name}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()

## 2 — Profile evolution vs. timestep (relaxation phase)

Each observable sampled as up to 10 curves **starting after** the 10 % compression
halt (step shown above).  Colour = timestep (cividis); the **bold black** curve is
the final (most-relaxed) profile.  Gel interior shaded.  Per the request, no extra
time-averaged curve is drawn, and in/out-gel means are written only when the final
curve is flat inside the gel (i.e. has equilibrated).

In [ ]:
# ==========================================================================
#  Profile evolution over the production HOLD (draining transient -> equilibrated plateau)
#  Row 1 : total sigma_zz, solvent mass density, solvent mass fraction
#  Row 2 : solvent PARTIAL stress (group), solvent-only vs partial (bin-by-bin),
#          solvent partial VIRIAL in energy units (non-volume; divide by the
#          Voronoi solvent volume later to get the true solvent-phase stress).
#  NOTE: every group-based profile below (total sigma_zz, sigma_s, sigma_p) already
#        carries the kinetic term n*kT -- `compute stress/atom NULL` does NOT drop
#        it (NULL suppresses only the velocity-bias temp compute; with no keywords
#        the compute defaults to `ke pair bond angle dihedral improper kspace fix`).
#        The pair/local sigma_ss is virial-only by construction, so ADD_KINETIC
#        (Config cell) adds n_s*kT to it before it is plotted here -- without that,
#        panel (e) would be comparing two different quantities.
#        With kinetic added, the REMAINING gap in (e) is (1/2)*W_sp, the
#        solvent-polymer cross virial: ~0 in the pure-solvent reservoir (nothing to
#        cross-interact with) and ~+0.26 inside the gel.  That gap is real physics
#        -- momentum handed off between the two phases -- not a broken
#        reconstruction, and it is why no volume normalisation of the solvent
#        stress alone yields the pore pressure (see the §2b header and §3).
# ==========================================================================
def post_halt(ts, stack):
    ts = np.asarray(ts); stack = np.asarray(stack)
    m = ts >= evol_from       # evolution PLOT: full hold (transient -> plateau)
    if m.sum() < 2:            # window past last snapshot -> use last few frames
        m = np.zeros(len(ts), bool); m[-min(len(ts), n_curves):] = True
    return subsample(ts[m], stack[m], n_curves)

def _plateau_mean(ts, stack):
    """Bin-wise mean over the equilibrated plateau (ts >= halt_ts); last frame if
    the window is empty.  Shared with section 2b."""
    ts = np.asarray(ts); stack = np.asarray(stack)
    m = np.asarray(ts) >= halt_ts
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        return np.nanmean(stack[m], axis=0) if m.any() else stack[-1]

# row-1 series
tzz_ts,  tzz_ev = post_halt(prod_ts, np.array(sig_t_zz))          # total sigma_zz
rho_ts,  rho_ev = post_halt(dens_ts, dens_m)                      # solvent mass density
mf_ts,   mf_ev  = post_halt(dens_ts, dens_m / rho_s0)             # solvent mass fraction

# row-2 series
szzs_ts, szzs_ev = post_halt(prod_ts, np.array(sig_s_zz))         # solvent PARTIAL stress (group)

# solvent-only (pair/local) vs partial (group) — plateau means on the same grid
grp_plateau = _plateau_mean(prod_ts, np.array(sig_s_zz))
ss_plateau  = None
if ss_prod_stack is not None and len(ss_prod_stack):
    ss_plateau = _plateau_mean(ss_prod_ts, ss_prod_stack)

# NON-VOLUME solvent zz virial (energy units) — written by the patched .lmp
# (fix avg_virialzz_solv -> virialzz_solvent_*.dat).  Absent on older runs.
# F_VIRZZ_S comes from the Config cell; section 2b divides it by V_solv.
vir_ts = vir_ev = None
if Path(F_VIRZZ_S).exists():
    _vz = read_ave_time_file(F_VIRZZ_S)
    if _vz:
        _vt = np.array([x[0] for x in _vz]); _vs = np.array([x[2] for x in _vz])
        vir_ts, vir_ev = post_halt(_vt, _vs)
        print(f'non-volume solvent virial loaded: {len(_vz)} frames from {F_VIRZZ_S.name}')
if vir_ev is None:
    print(f'NOTE: {F_VIRZZ_S.name} not found — panel (f) is a placeholder until you '
          're-run LAMMPS with the non-volume virial output (section 2b is skipped too).')

fig, axes = plt.subplots(2, 3, figsize=(23, 12), constrained_layout=True)
fig.suptitle(f'Profile evolution over the production hold \u2014 transient (steps >= {evol_from}) to equilibrated plateau (>= {halt_ts}):  {sim_name}',
             fontsize=14, fontweight='bold')

# ── Row 1 ────────────────────────────────────────────────────────────────
plot_evolution(axes[0,0], z_coords, tzz_ts, tzz_ev,
               r'$\sigma_{zz}^{t}(z,t)$', r'(a) Total stress $\sigma_{zz}^{t}$')
plot_evolution(axes[0,1], dens_z, rho_ts, rho_ev,
               r'$\rho_s(z,t)\ (m\,\sigma^{-3})$', r'(b) Solvent mass density $\rho_s$')
plot_evolution(axes[0,2], dens_z, mf_ts, mf_ev,
               r'$\rho_s/\rho_{s,0}$', r'(c) Solvent mass fraction $\rho_s/\rho_{s,0}$')
axes[0,2].axhline(1.0, color='k', ls=':', lw=1.2, alpha=0.6)

# ── Row 2 ────────────────────────────────────────────────────────────────
# (d) solvent PARTIAL stress (group-based; reliable)
plot_evolution(axes[1,0], z_coords, szzs_ts, szzs_ev,
               r'$\sigma_{s,zz}(z,t)$', r'(d) Solvent partial stress $\sigma_{s,zz}$ (group)')

# (e) solvent-only (pair/local) vs partial (group), bin-by-bin plateau means
axe = axes[1,1]
axe.plot(zn(z_coords), grp_plateau, '-', lw=2.6, color='steelblue',
         label=r'$\sigma_{s,zz}$ partial (group)')
if ss_plateau is not None:
    _ss_lab = (r'$\sigma_{s,zz}^{ss}$ (ss pair + kinetic)' if ADD_KINETIC
               else r'$\sigma_{s,zz}^{ss}$ (ss pair virial only)')
    axe.plot(zn(z_coords), ss_plateau, '-', lw=1.7, color='crimson', alpha=0.9,
             label=_ss_lab)
    # Drop the extreme-edge bin: the last chunk is only partly inside the box, so
    # its stress is diluted by the full V_bin divisor and would drag the reservoir
    # mean down (~-0.04) -- the same guard §3 applies to its baseline window.
    _zfe = zn(z_coords)
    _res = (_zfe < 0.10) | ((_zfe > 0.90) & (_zfe < 0.995))
    with np.errstate(invalid='ignore', divide='ignore'):
        _num = float(np.nanmean(ss_plateau[_res])); _den = float(np.nanmean(grp_plateau[_res]))
    _gap_note = ('  (cross virial; $\\approx 0$ expected, no polymer here)' if ADD_KINETIC
                 else '  $\\approx n_s kT$ (kinetic term, not yet added)')
    axe.text(0.02, 0.02, f'reservoir  ss = {_num:.2f}   group = {_den:.2f}   '
                         f'gap = {_den-_num:.2f}' + _gap_note,
             transform=axe.transAxes, fontsize=10, va='bottom',
             bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))
else:
    axe.text(0.5, 0.5, 'ss (pair/local) stack\nunavailable', ha='center', va='center',
             transform=axe.transAxes)
axe.axhline(0, color='k', ls='--', lw=1, alpha=0.4)
axe.set_xlim(0, 1); axe.grid(alpha=0.3)
axe.set_xlabel(r'$z/L_z$'); axe.set_ylabel(r'$\sigma_{s,zz}$ (LJ)')
axe.set_title(r'(e) Solvent-only vs partial (bin-by-bin)')
axe.legend(fontsize=10, loc='best')

# (f) NON-VOLUME solvent partial virial (energy units); divide by Voronoi V later
if vir_ev is not None:
    plot_evolution(axes[1,2], z_coords, vir_ts, vir_ev,
                   r'$W_{s,zz}(z,t)$  (energy)',
                   r'(f) Solvent partial virial $W_{s,zz}$  (÷ $V_{\rm solv}$ later)')
else:
    axes[1,2].text(0.5, 0.5, 'virialzz_solvent_*.dat not found\n'
                   '(re-run LAMMPS with the\nnon-volume virial output)',
                   ha='center', va='center', transform=axes[1,2].transAxes)
    axes[1,2].set_xlim(0, 1); axes[1,2].grid(alpha=0.3)
    axes[1,2].set_xlabel(r'$z/L_z$'); axes[1,2].set_ylabel(r'$W_{s,zz}$ (energy)')
    axes[1,2].set_title(r'(f) Solvent partial virial $W_{s,zz}$  (÷ $V_{\rm solv}$ later)')

out = PLOT_DIR / f'profile_evolution_{sim_name}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()


## 2b — Solvent-phase stress: $W_{s,zz}/V_{\rm solv}$  (diagnostic — *not* the pore pressure)

Panel (f) of §2 plots the raw non-volume solvent zz stress $W_{s,zz}(z)$ (energy units).
Dividing it by the **whole bin volume** gives back the group partial stress
$\sigma_{s,zz}$ of panel (d). Here it is divided by the **solvent's own partial
volume** instead, using two independent estimates of $\phi_s(z)$:

- **Option A — mass fraction:** $\phi_s^{\rm mf}=\rho_s(z)/\rho_{s,0}$. Cheap (density
  file only), but assumes the solvent keeps its bulk-reservoir specific volume.
- **Option B — Voronoi:** periodic voro++ tessellation of the trajectory, computed
  in the notebook (not in LAMMPS), summing solvent cell volumes per $z$-bin. Same
  `tess` library and mobile-atom convention as `longitudinal_modulus_analysis.ipynb`.

Requires `virialzz_solvent_*.dat` from the patched `triaxial_compression.lmp`.

> **What $W_{s,zz}$ actually contains.** `compute stress/atom NULL` does **not** drop
> the kinetic term — `NULL` only suppresses the velocity-bias temperature compute,
> and with no keywords the compute defaults to
> `ke pair bond angle dihedral improper kspace fix`. So
> $$W_{s,zz}/V_{\rm bin} \;=\; W_{ss}/V_{\rm bin} \;+\; n_s k_BT \;+\; \tfrac12 W_{sp}/V_{\rm bin},$$
> verified in the reservoir where solvent–wall coeffs are off and every solvent
> interaction is a 3–3 pair: group $=1.4872$, ss pair virial $=1.0136$,
> difference $=0.4737$ against $n_s k_BT = 0.4680$.
>
> **Neither option below is the pore pressure.** Two reasons:
> 1. The $\tfrac12 W_{sp}$ term is momentum handed off to the *network* ($\approx 0.26$
>    in-gel, larger than the ss part itself at $0.25$). Dividing it by a solvent
>    volume is meaningless.
> 2. Even the clean ss piece resists it: $W_{ss}/V_{\rm solv}$ is $0.58$ (mass-frac)
>    or $\approx0.74$ (Voronoi) in-gel against $1.014$ in the reservoir. If the solvent
>    inside really sat at bulk density in its own partial volume these would match.
>    They don't — the polymer genuinely suppresses solvent–solvent coordination, and
>    the missing repulsion is carried by solvent–polymer contacts. The deficit is not
>    geometric, so **no** choice of $\phi_s$ closes it.
>
> The ~12% in-gel/reservoir offset below is therefore a normalisation artefact, not
> compression physics: it is already **12%** in the $\varepsilon=0$ reference state
> (`*_ref` files, §1) and only reaches 15% at $\varepsilon=0.10$. For scale, the piston
> stress at $\varepsilon=0.10$ is $F_z/A = 0.060$, while the $\phi^{\rm mf}$-vs-$\phi^{\rm vor}$
> ambiguity alone moves the result by $0.21$ — 3.5× the entire signal.
>
> **Use §3 for the pore pressure instead.** It reads the baseline off the flat
> *reservoir*, where $\phi_s=1$ and there is no normalisation ambiguity at all. Keep
> this section as a diagnostic of $\phi_s(z)$ and of the cross-virial magnitude.

In [ ]:
# ==========================================================================
#  2b — TRUE SOLVENT-PHASE STRESS:  sigma^phase_s,zz(z) = W_s,zz(z) / V_solv(z)
#
#  DIAGNOSTIC ONLY -- this is NOT the pore pressure.  See the markdown above:
#  W_s,zz already contains n_s*kT AND half the solvent-polymer cross virial, and
#  no choice of phi_s makes the profile flat (the in-gel/reservoir offset is
#  already 12% at eps=0).  §3 gets p_pore from the flat reservoir instead.
#
#  W_s,zz(z) is the NON-VOLUME solvent zz stress written by the patched .lmp
#  (compute virialzz_solv -> virialzz_solvent_*.dat), in ENERGY units; panel (f)
#  of the previous cell plots it raw.  Dividing it by the WHOLE bin volume just
#  regenerates the group partial stress sigma_s,zz of panel (d), which is
#  diluted by the polymer/void space sharing the bin.  Rescaling to the
#  solvent's OWN partial volume gives:
#
#      sigma^phase_s,zz(z) = W_s,zz(z) / V_solv(z),    V_solv(z) = phi_s(z)*V_bin
#
#  Two independent estimates of phi_s(z) are computed here and compared:
#
#   OPTION A — MASS FRACTION (cheap; needs only the density file)
#       phi^mf_s(z) = rho_s(z) / rho_s,0
#     Treats the solvent as incompressible with the specific volume it has in
#     the bulk reservoir, so a local mass deficit is read as pure volume
#     exclusion.  Algebraically this is just sigma_s,zz(z)/phi_s(z).
#
#   OPTION B — VORONOI (exact geometry; computed HERE, not in LAMMPS)
#       phi^vor_s(z) = sum{ V_cell(i) : i solvent, z_i in bin } / V_bin
#     Full 3-D PERIODIC Voronoi tessellation of the trajectory via voro++
#     (`tess`), the same library and mobile-atom convention used by
#     longitudinal_modulus_analysis.ipynb.  Makes no incompressibility
#     assumption — it measures the space the polymer actually leaves the
#     solvent.  Monodisperse (plain, not radical) tessellation: all beads
#     are sigma = 1 here, so no radii weighting is needed.
#
#  Prerequisite: run cell §2 first (post_halt / _plateau_mean live there).
#  This cell re-reads the virial file itself, so it works even if §2 fell back
#  to the panel-(f) placeholder.
# ==========================================================================

# ---- knobs ---------------------------------------------------------------
VOR_ENABLE      = True    # False -> Option A only (skips the trajectory pass entirely)
VOR_MAX_FRAMES  = 4       # frames tessellated, evenly spaced over the window panel (f)
                          # plots.  ~20 s/frame at ~1.9e5 mobile atoms — raise for a
                          # smoother phi^vor, but the cost is linear.
VOR_MOBILE_ONLY = True    # tessellate types 1,2,3 only (matches
                          # longitudinal_modulus_analysis.ipynb); False -> all 5 types
VOR_NORM        = 'bin'   # 'bin'    -> phi_s = sum(V_cell solvent) / V_bin   [absolute]
                          # 'mobile' -> phi_s = sum(V_cell solvent) / sum(V_cell in bin)
                          #             (the longitudinal_modulus_analysis.ipynb
                          #             convention).  The two agree to <1% here because
                          #             the mobile phases fill the box; 'bin' is the one
                          #             that makes W/V_solv dimensionally honest.
PHI_FLOOR       = 0.02    # mask bins with phi_s below this — W/V_solv blows up there

# ---- W_s,zz: the non-volume solvent zz virial ----------------------------
_wv = read_ave_time_file(F_VIRZZ_S) if Path(F_VIRZZ_S).exists() else []
HAVE_W = bool(_wv)
if not HAVE_W:
    print(f'SKIP §2b: {F_VIRZZ_S.name} not found.\n'
          f'  Re-run triaxial_compression.lmp — it writes this via\n'
          f'  "fix avg_virialzz_solv ... c_virialzz_solv" (compute virialzz_solv).')
else:
    W_ts_all = np.array([x[0] for x in _wv])
    W_all    = np.array([x[2] for x in _wv])          # (nframes, nbins) energy units
    print(f'W_s,zz loaded: {W_all.shape[0]} frames x {W_all.shape[1]} bins from {F_VIRZZ_S.name}')

    # V_BIN / LX / LY / AREA_XY come from the load cell (one read_box() call).
    # sanity: W / V_bin must reproduce the group partial stress sigma_s,zz of panel (d)
    _k = int(np.argmin(np.abs(prod_ts - W_ts_all[-1])))
    _chk = np.nanmax(np.abs(W_all[-1]/V_BIN - np.asarray(sig_s_zz)[_k]))
    print(f'consistency check  max|W/V_bin - sigma_s,zz(group)| = {_chk:.3e}  '
          f'(should be ~0; large => binWidth or box mismatch)')

    # same time window / subsampling as panel (f)
    W_ts, W_ev = post_halt(W_ts_all, W_all)

    # ======================================================================
    #  OPTION A — mass-fraction volume fraction
    # ======================================================================
    def _phi_mf(ts):
        """phi_s(z) = rho_s(z)/rho_s,0 on z_coords, from the density frame nearest ts."""
        k = int(np.argmin(np.abs(dens_ts - ts)))
        return np.interp(z_coords, dens_z, dens_m[k]) / rho_s0, int(dens_ts[k])

    phi_mf_ev = np.array([_phi_mf(t)[0] for t in W_ts])
    _dt_match = max(abs(_phi_mf(t)[1] - t) for t in W_ts)
    print(f'Option A: density frames matched to virial frames, max |dt| = {_dt_match} steps')

    def _safe_div(W, phi):
        p = np.where(np.asarray(phi) < PHI_FLOOR, np.nan, phi)
        with np.errstate(invalid='ignore', divide='ignore'):
            return W / (p * V_BIN)

    sig_phase_mf_ev = _safe_div(W_ev, phi_mf_ev)

    # ======================================================================
    #  OPTION B — Voronoi volume fraction (voro++ via `tess`, periodic)
    # ======================================================================
    def _voronoi_V_solv(box, types, xyz):
        """Absolute solvent Voronoi volume per z-bin.
        Returns (z_centers, V_solv[nb], V_tot[nb], V_bin_k[nb]) on the frame's own
        grid (origin = box zlo, width = binWidth — identical to the LAMMPS chunks)."""
        import tess
        origin = np.array([box['x'][0], box['y'][0], box['z'][0]])
        L = np.array([box[k][1] - box[k][0] for k in 'xyz'])
        keep = np.isin(types, (1, 2, 3)) if VOR_MOBILE_ONLY else np.ones(len(types), bool)
        t_k = types[keep]
        p = origin + (xyz[keep] - origin) % L          # wrap into the primary box
        # voro++ fails on coincident points — drop exact duplicates (normally none)
        _, ui = np.unique(np.round(p, 6), axis=0, return_index=True)
        if len(ui) < len(p):
            print(f'    warning: dropping {len(p)-len(ui)} duplicate positions')
            ui = np.sort(ui); p, t_k = p[ui], t_k[ui]
        cont = tess.Container(p, limits=(tuple(origin), tuple(origin + L)), periodic=True)
        vol = np.array([c.volume() for c in cont])
        _err = abs(vol.sum() - L.prod()) / L.prod()
        if _err > 0.01:
            print(f'    warning: sum(V_cell) off box volume by {100*_err:.1f}%')
        nb = int(np.ceil(L[2] / binWidth))
        edges = np.minimum(origin[2] + np.arange(nb + 1) * binWidth, origin[2] + L[2])
        centers = 0.5 * (edges[:-1] + edges[1:])
        bi = np.clip(((p[:, 2] - origin[2]) / binWidth).astype(int), 0, nb - 1)
        s = (t_k == 3)
        V_s = np.bincount(bi[s], weights=vol[s], minlength=nb)[:nb]
        V_t = np.bincount(bi,    weights=vol,    minlength=nb)[:nb]
        return centers, V_s, V_t, L[0] * L[1] * np.diff(edges)

    phi_vor_ev = None; W_ts_vor = None; sig_phase_vor_ev = None
    if not VOR_ENABLE:
        print('Option B: VOR_ENABLE=False — skipped.')
    elif not Path(F_TRAJSTRESS).exists():
        print(f'Option B: {F_TRAJSTRESS.name} not found — skipped.')
    else:
        W_ts_vor, W_ev_vor = subsample(np.asarray(W_ts), np.asarray(W_ev), VOR_MAX_FRAMES)
        print(f'Option B: tessellating {len(W_ts_vor)} frame(s) {list(map(int, W_ts_vor))} '
              f'— expect ~20 s each ...')
        _fr = stream_traj_frames(F_TRAJSTRESS, W_ts_vor)
        _missing = [int(t) for t in W_ts_vor if int(t) not in _fr]
        if _missing:
            print(f'    NOTE: no traj frame at {_missing} — dropped '
                  f'(traj_stress_sync and the virial fix should share nfreq_stress)')
        _keep = [i for i, t in enumerate(W_ts_vor) if int(t) in _fr]
        if not _keep:
            print('Option B: no matching trajectory frames — skipped.')
            W_ts_vor = None
        else:
            W_ts_vor, W_ev_vor = np.asarray(W_ts_vor)[_keep], np.asarray(W_ev_vor)[_keep]
            _phis = []
            for t in W_ts_vor:
                box, typ, xyz = _fr[int(t)]
                zc, V_s, V_t, V_k = _voronoi_V_solv(box, typ, xyz)
                den = V_k if VOR_NORM == 'bin' else np.where(V_t > 0, V_t, np.nan)
                with np.errstate(invalid='ignore', divide='ignore'):
                    ph = V_s / den
                _phis.append(np.interp(z_coords, zc, ph, left=np.nan, right=np.nan))
                print(f'    ts {int(t)}: phi^vor max = {np.nanmax(ph):.3f}')
            phi_vor_ev = np.array(_phis)
            sig_phase_vor_ev = _safe_div(W_ev_vor, phi_vor_ev)
            del _fr

    # ---- plateau means (_plateau_mean is defined in section 2) --------------
    phi_mf_plateau   = _plateau_mean(W_ts, phi_mf_ev)
    sig_mf_plateau   = _plateau_mean(W_ts, sig_phase_mf_ev)
    sig_bin_plateau  = _plateau_mean(W_ts, W_ev / V_BIN)          # = group sigma_s,zz
    phi_vor_plateau  = _plateau_mean(W_ts_vor, phi_vor_ev)       if phi_vor_ev is not None else None
    sig_vor_plateau  = _plateau_mean(W_ts_vor, sig_phase_vor_ev) if phi_vor_ev is not None else None

    _z_pist = piston_z_at(np.asarray(W_ts)[-1])
    interior = in_gel & (z_coords >= z_gel_lo + wall_margin) & (z_coords <= _z_pist - wall_margin)

    # ======================================================================
    #  PLOTS
    # ======================================================================
    fig, axes = plt.subplots(2, 2, figsize=(18, 13), constrained_layout=True)
    fig.suptitle(r'§2b  Solvent-phase stress  $\sigma^{\rm phase}_{s,zz}=W_{s,zz}/V_{\rm solv}$'
                 f'  —  mass-fraction vs Voronoi:  {sim_name}',
                 fontsize=14, fontweight='bold')

    # (a) the two volume fractions, plateau means
    ax = axes[0, 0]
    ax.plot(zn(z_coords), phi_mf_plateau, '-', lw=2.6, color=WONG['blue'],
            label=r'(A) $\phi_s^{\rm mf}=\rho_s/\rho_{s,0}$')
    if phi_vor_plateau is not None:
        ax.plot(zn(z_coords), phi_vor_plateau, '-', lw=2.2, color=WONG['vermillion'],
                label=r'(B) $\phi_s^{\rm vor}$ (Voronoi)')
        _r = np.nanmean(phi_vor_plateau[interior] / phi_mf_plateau[interior]) if interior.any() else np.nan
        ax.text(0.02, 0.03, f'in-gel $\\phi^{{\\rm vor}}/\\phi^{{\\rm mf}}$ = {_r:.3f}',
                transform=ax.transAxes, va='bottom', fontsize=14,
                bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))
    ax.axhline(1.0, color='k', ls=':', lw=1.2, alpha=0.6)
    ax.axhline(PHI_FLOOR, color='r', ls=':', lw=1.0, alpha=0.5)
    shade_gel(ax); mark_walls(ax, W_ts)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.15); ax.grid(alpha=0.3)
    ax.set_xlabel(r'$z/L_z$'); ax.set_ylabel(r'$\phi_s$')
    ax.set_title('(a) Solvent volume fraction (plateau mean)')
    ax.legend(fontsize=13, loc='best')

    # (b) Option A evolution
    plot_evolution(axes[0, 1], z_coords, W_ts, sig_phase_mf_ev,
                   r'$\sigma^{\rm phase}_{s,zz}$ (LJ)',
                   r'(b) Option A — $W_{s,zz}/(\phi_s^{\rm mf}V_{\rm bin})$')
    robust_ylim(axes[0, 1], sig_phase_mf_ev, zmask=interior)

    # (c) Option B evolution
    ax = axes[1, 0]
    if sig_phase_vor_ev is not None:
        plot_evolution(ax, z_coords, W_ts_vor, sig_phase_vor_ev,
                       r'$\sigma^{\rm phase}_{s,zz}$ (LJ)',
                       r'(c) Option B — $W_{s,zz}/V^{\rm vor}_{\rm solv}$'
                       f'  [{len(W_ts_vor)} frames]')
        robust_ylim(ax, sig_phase_vor_ev, zmask=interior)
    else:
        ax.text(0.5, 0.5, 'Voronoi option unavailable\n(VOR_ENABLE=False or\ntraj_stress dump missing)',
                ha='center', va='center', transform=ax.transAxes)
        ax.set_xlim(0, 1); ax.grid(alpha=0.3)
        ax.set_xlabel(r'$z/L_z$'); ax.set_ylabel(r'$\sigma^{\rm phase}_{s,zz}$ (LJ)')
        ax.set_title(r'(c) Option B — $W_{s,zz}/V^{\rm vor}_{\rm solv}$')

    # (d) plateau-mean comparison of all three normalisations
    ax = axes[1, 1]
    _curves = [sig_bin_plateau, sig_mf_plateau]
    ax.plot(zn(z_coords), sig_bin_plateau, '-', lw=2.6, color='seagreen',
            label=r'$W/V_{\rm bin}$ = $\sigma_{s,zz}$ (group, diluted)')
    ax.plot(zn(z_coords), sig_mf_plateau, '-', lw=2.4, color=WONG['blue'],
            label=r'(A) $W/(\phi_s^{\rm mf}V_{\rm bin})$')
    if sig_vor_plateau is not None:
        ax.plot(zn(z_coords), sig_vor_plateau, '-', lw=2.4, color=WONG['vermillion'],
                label=r'(B) $W/V^{\rm vor}_{\rm solv}$')
        _curves.append(sig_vor_plateau)
    if ss_plateau is not None:
        ax.plot(zn(z_coords), ss_plateau, '--', lw=1.6, color=WONG['green'], alpha=0.9,
                label=(r'$\sigma^{ss}_{s,zz}$ (ss pair + kinetic)' if ADD_KINETIC
                       else r'$\sigma^{ss}_{s,zz}$ (ss pair virial only)'))
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
    shade_gel(ax); mark_walls(ax, W_ts)
    ax.set_xlim(0, 1); ax.grid(alpha=0.3)
    ax.set_xlabel(r'$z/L_z$'); ax.set_ylabel(r'$\sigma_{s,zz}$ (LJ)')
    ax.set_title('(d) Plateau means: three volume normalisations')
    ax.legend(fontsize=11, loc='best')
    robust_ylim(ax, _curves, zmask=interior)

    out = PLOT_DIR / f'solvent_phase_stress_{sim_name}{LEVEL_TAG}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()

    # ---- in-gel summary ----------------------------------------------------
    print('\nin-gel plateau means (interior, wall_margin trimmed):')
    print(f'  phi_s   mass-fraction = {_fmt_mu(phi_mf_plateau[interior])}')
    if phi_vor_plateau is not None:
        print(f'  phi_s   Voronoi       = {_fmt_mu(phi_vor_plateau[interior])}')
    print(f'  sigma   W/V_bin (group) = {_fmt_mu(sig_bin_plateau[interior])}')
    print(f'  sigma   (A) mass-frac   = {_fmt_mu(sig_mf_plateau[interior])}')
    if sig_vor_plateau is not None:
        print(f'  sigma   (B) Voronoi     = {_fmt_mu(sig_vor_plateau[interior])}')

### Optional — polymer-only stress σ_p,pp evolution

Provided separately since it is the new observable.  Same conventions as above.
If σ_p,pp looks sign-flipped relative to the group polymer stress (see the
validation cell), set `BOND_SIGN = -1.0` in Config and re-run.

In [ ]:
if pp_prod_stack is not None and len(pp_prod_stack):
    fig, axes = plt.subplots(1, 2, figsize=(17, 6), constrained_layout=True)
    if pp_ref_m is not None:
        plot_reference(axes[0], z_coords, pp_ref_m, pp_ref_lo, pp_ref_hi,
                       WONG['orange'], r'$\sigma_{p,zz}^{pp}(z)$',
                       r'Reference $\sigma_{p,zz}^{pp}$ (pair + bond'
                       + (' + kinetic)' if ADD_KINETIC else ', virial only)'))
    else:
        axes[0].text(0.5,0.5,'pp reference unavailable',ha='center',va='center',transform=axes[0].transAxes)
    pp_ts2, pp_ev = post_halt(pp_prod_ts, pp_prod_stack)
    plot_evolution(axes[1], z_coords, pp_ts2, pp_ev,
                   r'$\sigma_{p,zz}^{pp}(z,t)$', r'Evolution $\sigma_{p,zz}^{pp}$')
    out = PLOT_DIR / f'polymer_only_stress_{sim_name}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
else:
    print('sigma_p,pp production data unavailable - skipping.')

## 3 — Network (effective) stress & pore pressure

Terzaghi decomposition $\sigma^{t}_{zz}=\sigma'_{zz}+p_{\rm pore}$.  The **network stress** $\sigma'_{zz}$ is the jump of the total stress above the pore-fluid baseline (the barostat pressure, read per-curve from the flat reservoir at $z/L_z\approx0.95$); the **pore pressure** is total $-$ network.  Colour = timestep, **bold black** = final (equilibrated) curve; the first (fast-piston) curve is handled with fixed membrane bounds and a fixed baseline window. 95% error bands on every curve.

In [ ]:
# ==========================================================================
#  3 — NETWORK (EFFECTIVE) STRESS  σ'_zz  &  PORE PRESSURE  (evolution)
# ==========================================================================
# Terzaghi decomposition of the total normal stress inside the membrane:
#       σ_zz^t(z,t) = σ'_zz(z,t) + p_pore(z,t)
# The network (effective) stress is the JUMP of the total stress above the
# pore-fluid baseline; the pore pressure is then total − network.
#
# Baseline (pore) pressure is read PER CURVE from the flat far-reservoir around
# z/Lz = BASELINE_ZF — this is the barostat/bath pressure (≈ P_target = 1.5 for
# a P=1.5 run).  z/Lz≈0.95 stays flat even for the fast-piston FIRST curve
# (the piston slope sits between the membrane and this window), so the
# subtraction is clean for every timestep.
#
# FIXED bounds (per the request): the membrane extent is determined ONCE from
# the final equilibrated polymer stress and reused for ALL timesteps, so the
# fast piston in the first curve cannot distort a per-curve gel detection.
BASELINE_ZF      = 0.95     # fractional z (z/Lz) of the reservoir baseline window
BASELINE_ZF_HALF = 0.04     # half-width of that window (fraction of Lz)

zf = (z_coords - z_coords.min()) / (z_coords.max() - z_coords.min())
_bw = (zf >= BASELINE_ZF - BASELINE_ZF_HALF) & (zf <= BASELINE_ZF + BASELINE_ZF_HALF)
_bw &= (zf < 0.995)          # drop the half-empty extreme-edge bin (stress rolls off)
print(f'pore baseline window: z/Lz in [{BASELINE_ZF-BASELINE_ZF_HALF:.2f}, '
      f'{BASELINE_ZF+BASELINE_ZF_HALF:.2f}]  ({_bw.sum()} bins, '
      f'z in [{z_coords[_bw].min():.0f}, {z_coords[_bw].max():.0f}])')

# ---- FIXED membrane bounds from the final (equilibrated) polymer stress ----
# Piston is out of the membrane by the final curve, so this is the clean extent.
_spf = np.abs(sig_p_zz[-1]); _thr = gel_thresh * float(np.nanmax(_spf))
_mem = _spf > _thr
z_mem_lo = float(z_coords[_mem].min()) if _mem.any() else z_gel_lo
z_mem_hi = float(z_coords[_mem].max()) if _mem.any() else z_gel_hi
in_mem = (z_coords >= z_mem_lo) & (z_coords <= z_mem_hi)
print(f'membrane (FIXED for all curves, from final polymer stress): '
      f'z in [{z_mem_lo:.1f}, {z_mem_hi:.1f}]  ({in_mem.sum()} bins)')

# ---- per-bin single-snapshot noise, estimated from the reference window ----
# A single fix ave/time vector carries no per-bin variance; the reference
# snapshots share the same nfreq averaging, so their per-bin scatter is a fair
# proxy for the statistical noise of ONE production σ_zz^t snapshot.
sd_bin = np.nanstd(ref_t_stack, axis=0)
# Fallback when the reference window is too thin (<2 snapshots) to estimate
# per-bin scatter: use the spatial scatter of the total stress in the flat
# reservoir baseline window as a uniform noise floor so error bars never vanish.
if ref_t_stack.shape[0] < 2 or not np.any(sd_bin > 0):
    _floor = float(np.nanstd(np.asarray(sig_t_zz)[-1][_bw]))
    sd_bin = np.full(len(z_coords), _floor)
    print(f'  (thin reference: using reservoir noise floor sd_bin = {_floor:.4f})')
_z95   = 1.959963985                                  # 95% normal quantile

# ---- network stress + pore pressure for every production snapshot ----
tot_stack = np.asarray(sig_t_zz)                      # (n_prod, nz)
net_stack = np.zeros_like(tot_stack)
pore_val  = np.zeros(len(tot_stack))                  # scalar baseline per curve
pore_half = np.zeros(len(tot_stack))                  # 95% half-width of the baseline
for i in range(len(tot_stack)):
    v  = tot_stack[i][_bw]; v = v[np.isfinite(v)]
    p0 = float(np.nanmean(v))
    se = float(stats.sem(v)) if len(v) > 1 else 0.0
    tcr = stats.t.ppf(0.5 + ci_level/2, df=max(len(v)-1, 1))
    pore_val[i]  = p0
    pore_half[i] = tcr * se
    net_stack[i] = tot_stack[i] - p0                  # σ'_zz = σ^t − p_pore
# network 95% band: per-bin snapshot noise ⊕ baseline uncertainty (in quadrature)
net_half = np.sqrt((_z95 * sd_bin[None, :])**2 + (pore_half[:, None])**2)

print(f"pore pressure (baseline)  t={prod_ts[0]}→{prod_ts[-1]}: "
      f"{pore_val[0]:.3f} → {pore_val[-1]:.3f}")
print(f"network σ'_zz in membrane t={prod_ts[0]}→{prod_ts[-1]}: "
      f"{np.nanmean(net_stack[0][in_mem]):.3f} → {np.nanmean(net_stack[-1][in_mem]):.3f}")


In [ ]:
# ---- evolution plots: network (effective) stress + pore pressure ----
# Same conventions as the other evolution plots: colour = timestep (cividis),
# the bold black curve is the final (equilibrated) profile, membrane shaded.
# The FIRST curve is the fast-piston / actively-compressed state; because the
# membrane bounds and the z/Lz≈0.95 baseline are fixed, it is handled cleanly.
def _shade_mem(ax):
    ax.axvspan(zn(z_mem_lo), zn(z_mem_hi), **GEL_SHADE)

net_ts, net_ev  = subsample(prod_ts, net_stack, n_curves)
_,      neth_ev = subsample(prod_ts, net_half,  n_curves)
_,      porev   = subsample(prod_ts, pore_val,  n_curves)
_,      poreh   = subsample(prod_ts, pore_half, n_curves)

fig, axes = plt.subplots(1, 2, figsize=(17, 6), constrained_layout=True)
fig.suptitle(f'Network stress & pore pressure evolution:  {sim_name}',
             fontsize=14, fontweight='bold')
norm = Normalize(vmin=net_ts.min(), vmax=net_ts.max()); cmap = plt.get_cmap(EVO_CMAP)
zc = zn(z_coords)

# (a) network (effective) stress σ'_zz(z,t)
axN = axes[0]
for i in range(len(net_ts)):
    last = (i == len(net_ts) - 1)
    c = 'k' if last else cmap(norm(net_ts[i])); lw = 3.5 if last else 1.6
    axN.fill_between(zc, net_ev[i]-neth_ev[i], net_ev[i]+neth_ev[i],
                     color=c, alpha=(0.20 if last else 0.06), lw=0, zorder=(4 if last else 2))
    axN.plot(zc, net_ev[i], '-', color=c, lw=lw,
             alpha=(1.0 if last else 0.8), zorder=(5 if last else 3))
axN.axhline(0, color='k', ls='--', lw=1, alpha=0.5); _shade_mem(axN); mark_walls(axN, net_ts)
axN.set_xlabel(r'$z/L_z$'); axN.set_ylabel(r"$\sigma'_{zz}(z,t)$")
axN.set_title(r"(a) Network stress $\sigma'_{zz}=\sigma^{t}_{zz}-p_{\mathrm{pore}}$")
axN.set_xlim(0, 1); axN.grid(alpha=0.3)
# y-limits framed to the membrane interior + reservoir (drop the support-side
# wall spike; percentiles absorb the piston-interface spike) so the ~0.8 membrane
# value and the ~0 reservoir are both visible instead of being clipped.
_net_ymask = (z_coords >= z_mem_lo + wall_margin) & (z_coords <= z_coords.max())
robust_ylim(axN, list(net_ev), zmask=_net_ymask, pad=0.15)
axN.text(0.02, 0.03, 'final (equilibrated)\n'
         f'mean in membrane = {_fmt_mu(net_ev[-1][in_mem])}',
         transform=axN.transAxes, va='bottom', ha='left', fontsize=14,
         bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))

# (b) pore pressure p_pore(z,t) = total − network = flat baseline
axP = axes[1]
_zends = [0.0, 1.0]
for i in range(len(net_ts)):
    last = (i == len(net_ts) - 1)
    c = 'k' if last else cmap(norm(net_ts[i])); lw = 3.5 if last else 1.6
    axP.fill_between(_zends, [porev[i]-poreh[i]]*2, [porev[i]+poreh[i]]*2,
                     color=c, alpha=(0.20 if last else 0.06), lw=0, zorder=(4 if last else 2))
    axP.plot(_zends, [porev[i]]*2, '-', color=c, lw=lw,
             alpha=(1.0 if last else 0.8), zorder=(5 if last else 3))
_shade_mem(axP); mark_walls(axP, net_ts)
axP.set_xlabel(r'$z/L_z$'); axP.set_ylabel(r'$p_{\mathrm{pore}}(z,t)$')
axP.set_title(r"(b) Pore pressure $p_{\mathrm{pore}}=\sigma^{t}_{zz}-\sigma'_{zz}$")
axP.set_xlim(0, 1); axP.grid(alpha=0.3)
axP.text(0.02, 0.03, 'final (equilibrated)\n'
         f'$p_{{\\mathrm{{pore}}}}$ = {_fmt_val_unc(porev[-1], poreh[-1])}',
         transform=axP.transAxes, va='bottom', ha='left', fontsize=14,
         bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))

sm = plt.cm.ScalarMappable(cmap=EVO_CMAP, norm=norm); sm.set_array([])
cb = fig.colorbar(sm, ax=axes, fraction=0.046, pad=0.02); cb.set_label('timestep')

out = PLOT_DIR / f'network_stress_pore_pressure_{sim_name}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()


## 4 — Piston pressure history (relaxation check)

$P(t) = F_{z,\mathrm{piston}}(t)/A$ from the pairwise contact force on the piston
atoms recorded by `fix out_piston_force` in `triaxial_compression.lmp` (same
methodology as `slab_with_flow.lmp` compression mode).  The cross-section
$A = l_x l_y$ is fixed in Phase 2 (no lateral barostat), so it is read once from
the pair-dump box header.  The **log panel** makes it easy to see when the
network has relaxed: once $\ln P$ flattens, the piston pressure has reached its
equilibrium plateau.

In [ ]:
# ==========================================================================
#  4 — PISTON FORCE / PRESSURE  (from out_piston_force in triaxial_compression.lmp)
# ==========================================================================
# The .lmp records the summed pairwise z-force on the piston atoms:
#     fix out_piston_force all print ... '$(step) $(c_piston_fz)' ...
#         file .../piston_force_<sim_name>.dat
# Pressure P = F_z / A, with A = AREA_XY the FIXED box cross-section (no lateral
# barostat in Phase 2), computed once in the load cell from the dump box header.
# F_PISTON_FORCE defined in the Config cell (also used by the Expanse sync cell).
piston_area = AREA_XY
print(f'piston area  A = lx*ly = {LX:.2f} x {LY:.2f} = {piston_area:.2f} sigma^2')

if Path(F_PISTON_FORCE).exists():
    _pf         = read_print_file(F_PISTON_FORCE, col_names=['step', 'F_piston_z'])
    steps_pf    = _pf['step'].astype(int)
    F_piston    = _pf['F_piston_z']
    P_piston_ts = F_piston / piston_area
    has_piston_force = True
    print(f'piston force: {len(steps_pf)} snapshots, steps '
          f'{steps_pf[0]} -> {steps_pf[-1]};  F_z in '
          f'[{F_piston.min():.2f}, {F_piston.max():.2f}] LJ;  '
          f'P in [{P_piston_ts.min():.4f}, {P_piston_ts.max():.4f}] LJ/sigma^2')
else:
    has_piston_force = False
    print(f'MISSING: {F_PISTON_FORCE.name} -- sync it from the cluster '
          '(output_files/piston_data/).  Piston-pressure plots will be skipped.')

# ---- LMP block-averaged piston force (optional; from fix out_piston_force_avg) ----
# Present only for runs made with the updated .lmp.  If absent, the plot below
# still smooths the raw series with a notebook-side rolling mean.
if Path(F_PISTON_FORCE_AVG).exists():
    _pfa            = read_print_file(F_PISTON_FORCE_AVG, col_names=['step', 'F_piston_z'])
    steps_pf_avg    = _pfa['step'].astype(int)
    F_piston_avg    = _pfa['F_piston_z']
    P_piston_avg_ts = F_piston_avg / piston_area
    has_piston_force_avg = True
    print(f'LMP block-averaged piston force: {len(steps_pf_avg)} snapshots '
          f'(pf_nevery x pf_nrepeat window)')
else:
    has_piston_force_avg = False
    print('LMP block-averaged piston force: not present '
          '(will smooth raw series with a rolling mean instead)')


In [ ]:
# ---- piston pressure P = F_z/A vs timestep: linear + log ----
# Raw c_piston_fz is thermally noisy (there is NO damping on the piston, by
# design).  We reduce the visual noise by TIME-AVERAGING, not damping:
#   - a notebook-side centered rolling mean of the raw series (always shown), and
#   - the LMP block-averaged series if the run has it (has_piston_force_avg).
# rolling_mean() lives in the readers cell (shared with the sweep histories).
roll_win = 21   # rolling-mean window in samples (odd; ~ roll_win*volume_freq steps)

if has_piston_force:
    peak_i      = int(P_piston_ts.argmax())
    peak_P_step = int(steps_pf[peak_i])
    P_roll      = rolling_mean(P_piston_ts, roll_win)   # smoothed raw pressure

    fig, (axL, axG) = plt.subplots(1, 2, figsize=(17, 6), constrained_layout=True)
    fig.suptitle(f'Piston pressure history:  {sim_name}\n'
                 f'$A = l_x\\,l_y = {piston_area:.1f}\\,\\sigma^2$',
                 fontsize=14, fontweight='bold')

    for ax in (axL, axG):
        ax.axvline(peak_P_step, color=WONG['blue'], ls='--', lw=1.6, alpha=0.7,
                   label=f'peak $P$ / relaxation start (t={peak_P_step})')
        ax.set_xlabel('step'); ax.grid(alpha=0.3)

    # (a) linear -----------------------------------------------------------
    axL.plot(steps_pf, P_piston_ts, '-', color=WONG['vermillion'], lw=1.0, alpha=0.30,
             label=r'$P = F_z/A$ (raw)')
    axL.plot(steps_pf, P_roll, '-', color=WONG['vermillion'], lw=2.6, alpha=0.95,
             label=f'rolling mean ({roll_win} pts)')
    if has_piston_force_avg:
        axL.plot(steps_pf_avg, P_piston_avg_ts, '-', color=WONG['black'], lw=1.8,
                 alpha=0.8, label='LMP block-avg')
    axL.axhline(0, color='k', ls='--', lw=0.8, alpha=0.4)
    axL.set_ylabel(r'$P = F_z/A$  (LJ / $\sigma^2$)')
    axL.set_title('(a) piston pressure')
    axL.legend(fontsize=13, loc='best')

    # (b) log  -- guard non-positive values (occur before the piston contacts gel)
    _pos = P_piston_ts > 0
    axG.plot(steps_pf[_pos], np.log(P_piston_ts[_pos]), '-',
             color=WONG['vermillion'], lw=1.0, alpha=0.30, label=r'$\ln(F_z/A)$ (raw)')
    _posr = P_roll > 0
    axG.plot(steps_pf[_posr], np.log(P_roll[_posr]), '-',
             color=WONG['vermillion'], lw=2.6, alpha=0.95,
             label=f'rolling mean ({roll_win} pts)')
    if has_piston_force_avg:
        _posa = P_piston_avg_ts > 0
        axG.plot(steps_pf_avg[_posa], np.log(P_piston_avg_ts[_posa]), '-',
                 color=WONG['black'], lw=1.8, alpha=0.8, label='LMP block-avg')
    axG.set_ylabel(r'$\ln P$')
    axG.set_title('(b) log piston pressure (relaxation view)')
    axG.legend(fontsize=13, loc='best')

    out_P = PLOT_DIR / f'piston_pressure_history_{sim_name}.png'
    plt.savefig(out_P, dpi=150, bbox_inches='tight'); print('saved', out_P); plt.show()

    print(f'  peak  P = {P_piston_ts.max():.4f} LJ/sigma^2  at step {peak_P_step}')
    print(f'  final P (raw)         = {P_piston_ts[-1]:.4f} LJ/sigma^2  at step {steps_pf[-1]}')
    print(f'  final P (rolling {roll_win}) = {P_roll[-1]:.4f} LJ/sigma^2')
else:
    print('skipped (no piston_force file)')


## 4b — Piston force history (relaxation check)

$F_{z,\mathrm{piston}}(t)$: the raw summed pairwise z-force on the piston atoms
from `fix out_piston_force` in `triaxial_compression.lmp` (same series that
divided by $A=l_xl_y$ gives the pressure above).  Same format as the piston
pressure panels: raw + notebook rolling mean (+ LMP block-avg if present), peak
marked, with a **log panel** so the relaxation plateau is easy to read once
$\ln F_z$ flattens.

In [ ]:
# ---- piston force F_z vs timestep: linear + log ----
# Same format as the piston-pressure cell above, but plotting the raw force
# F_z (not F_z/A).  Reuses roll_win and rolling_mean() defined in the pressure
# cell.  Raw c_piston_fz is thermally noisy (no piston damping, by design), so
# we reduce visual noise by TIME-AVERAGING: a centered rolling mean of the raw
# series, and the LMP block-averaged series if the run has it.
if has_piston_force:
    peakF_i      = int(F_piston.argmax())
    peakF_step   = int(steps_pf[peakF_i])
    F_roll       = rolling_mean(F_piston, roll_win)      # smoothed raw force

    fig, (axL, axG) = plt.subplots(1, 2, figsize=(17, 6), constrained_layout=True)
    fig.suptitle(f'Piston force history:  {sim_name}\n'
                 f'$F_z$ = summed pairwise z-force on piston atoms',
                 fontsize=14, fontweight='bold')

    for ax in (axL, axG):
        ax.axvline(peakF_step, color=WONG['blue'], ls='--', lw=1.6, alpha=0.7,
                   label=f'peak $F_z$ / relaxation start (t={peakF_step})')
        ax.set_xlabel('step'); ax.grid(alpha=0.3)

    # (a) linear -----------------------------------------------------------
    axL.plot(steps_pf, F_piston, '-', color=WONG['vermillion'], lw=1.0, alpha=0.30,
             label=r'$F_z$ (raw)')
    axL.plot(steps_pf, F_roll, '-', color=WONG['vermillion'], lw=2.6, alpha=0.95,
             label=f'rolling mean ({roll_win} pts)')
    if has_piston_force_avg:
        axL.plot(steps_pf_avg, F_piston_avg, '-', color=WONG['black'], lw=1.8,
                 alpha=0.8, label='LMP block-avg')
    axL.axhline(0, color='k', ls='--', lw=0.8, alpha=0.4)
    axL.set_ylabel(r'$F_z$  (LJ units)')
    axL.set_title('(a) piston force')
    axL.legend(fontsize=13, loc='best')

    # (b) log  -- guard non-positive values (occur before the piston contacts gel)
    _pos = F_piston > 0
    axG.plot(steps_pf[_pos], np.log(F_piston[_pos]), '-',
             color=WONG['vermillion'], lw=1.0, alpha=0.30, label=r'$\ln F_z$ (raw)')
    _posr = F_roll > 0
    axG.plot(steps_pf[_posr], np.log(F_roll[_posr]), '-',
             color=WONG['vermillion'], lw=2.6, alpha=0.95,
             label=f'rolling mean ({roll_win} pts)')
    if has_piston_force_avg:
        _posa = F_piston_avg > 0
        axG.plot(steps_pf_avg[_posa], np.log(F_piston_avg[_posa]), '-',
                 color=WONG['black'], lw=1.8, alpha=0.8, label='LMP block-avg')
    axG.set_ylabel(r'$\ln F_z$')
    axG.set_title('(b) log piston force (relaxation view)')
    axG.legend(fontsize=13, loc='best')

    out_F = PLOT_DIR / f'piston_force_history_{sim_name}.png'
    plt.savefig(out_F, dpi=150, bbox_inches='tight'); print('saved', out_F); plt.show()

    print(f'  peak  F_z = {F_piston.max():.2f} LJ  at step {peakF_step}')
    print(f'  final F_z (raw)         = {F_piston[-1]:.2f} LJ  at step {steps_pf[-1]}')
    print(f'  final F_z (rolling {roll_win}) = {F_roll[-1]:.2f} LJ')
else:
    print('skipped (no piston_force file)')


## 4c — Piston–gel contact: where the piston meets the fuzzy boundary

Tracks the descending piston face against two definitions of the gel top edge —
the **fuzzy** outermost-strand edge (bounding-box max-$z$) and the robust
**body** edge (Rg-based) — alongside the piston force, so the force lift-off
reads directly as a contact gap / penetration depth into the fuzzy layer.

In [ ]:
# ==========================================================================
#  4c — PISTON-GEL CONTACT: how close the piston gets before the force rises
# ==========================================================================
# Two definitions of the gel TOP edge for the DETAIL level (both = COM_z + half
# the z-thickness):
#   FUZZY  z_top_bb = COM_z + 1/2 L_z^bb   -- outermost polymer bead (max-z).
#          The .lmp notes c_maxz is set by stray strands reaching into the
#          solvent, so this is the first thing the piston can touch.
#   BODY   z_top_rg = COM_z + 1/2 L_z^rg   -- robust Rg-based surface (outlier-
#          insensitive), i.e. the bulk gel top.
# Piston face = piston COM z (thin rigid sheet).  Contact gap:
#   gap(t) = z_piston(t) - z_top(t)      (>0 above the edge, <0 penetrating).
# (a) time series: descending piston + both edges (left axis) with the smoothed
#     piston force (right axis).  (b) force vs gap-to-edge: the force lift-off
#     point reads directly as a penetration depth into the fuzzy layer.

def _loadcols(path, mincols):
    if not Path(path).exists():
        return None
    a = np.atleast_2d(np.loadtxt(path, comments='#'))
    return a if (a.ndim == 2 and a.shape[1] >= mincols) else None

_bb  = _loadcols(DATA_DIR / f'gel_dimensions_bb_{sim_name}{LEVEL_TAG}.dat', 4)   # step lx ly lz
_rg  = _loadcols(DATA_DIR / f'gel_dimensions_rg_{sim_name}{LEVEL_TAG}.dat', 4)   # step lx ly lz
_com = _loadcols(DATA_DIR / f'polymer_com_{sim_name}{LEVEL_TAG}.dat', 4)         # step  x  y  z
_pos = _loadcols(F_PISTON_POS, 2)                                               # step  z

if (_bb is None) or (_com is None) or (_pos is None) or (not has_piston_force):
    print('4c skipped: need gel_dimensions_bb, polymer_com, piston_position and '
          f'piston_force for the detail level {LEVEL_TAG}.')
else:
    # bb / com / pos share strain_freq steps; align everything to the bb grid.
    s_edge   = _bb[:, 0].astype(float)
    com_z    = np.interp(s_edge, _com[:, 0], _com[:, 3])
    z_top_bb = com_z + 0.5 * _bb[:, 3]                       # fuzzy (outermost bead)
    z_pist   = np.interp(s_edge, _pos[:, 0], _pos[:, 1])     # piston face z(t)
    gap_bb   = z_pist - z_top_bb
    z_top_rg = gap_rg = None
    if _rg is not None:
        z_top_rg = com_z + 0.5 * np.interp(s_edge, _rg[:, 0], _rg[:, 3])   # body (Rg)
        gap_rg   = z_pist - z_top_rg

    # force on its own (coarser) grid -> smoothed; interp gap onto it for panel (b)
    gap_bb_f = np.interp(steps_pf, s_edge, gap_bb)
    gap_rg_f = np.interp(steps_pf, s_edge, gap_rg) if gap_rg is not None else None
    Fsm      = F_roll                                        # rolling-mean force (cell 4b)

    # contact onset = first time (piston descending) the smoothed force lifts off
    thr = max(0.05 * float(np.nanmax(Fsm)), 1.0)
    _hit = np.where(Fsm > thr)[0]
    onset_i = int(_hit[0]) if len(_hit) else None
    gap_bb_onset = float(gap_bb_f[onset_i]) if onset_i is not None else np.nan
    gap_rg_onset = float(gap_rg_f[onset_i]) if (onset_i is not None and gap_rg_f is not None) else np.nan

    fig, (axA, axB) = plt.subplots(1, 2, figsize=(18, 6.5), constrained_layout=True)
    fig.suptitle(f'Piston–gel contact (level {LEVEL_TAG.lstrip("_c") or COMP_LEVEL}):  {sim_name}',
                 fontsize=14, fontweight='bold')

    # (a) positions + force -------------------------------------------------
    lA = axA.plot(s_edge, z_pist,   color=WONG['black'],      lw=2.4, label=r'piston face $z_\mathrm{piston}$')
    lA += axA.plot(s_edge, z_top_bb, color=WONG['vermillion'], lw=2.0, label=r'fuzzy edge  (BB max-$z$)')
    if z_top_rg is not None:
        lA += axA.plot(s_edge, z_top_rg, color=WONG['blue'], lw=2.0, ls='--', label=r'body edge  (Rg)')
    axA.set_xlabel('step'); axA.set_ylabel(r'$z$  ($\sigma$)'); axA.grid(alpha=0.3)
    axA.set_title('(a) piston vs gel top edges')
    axF = axA.twinx()
    lF = axF.plot(steps_pf, Fsm, color=WONG['green'], lw=1.9, alpha=0.85,
                  label=r'$F_z$ (smoothed)')
    axF.set_ylabel(r'$F_z$  (LJ)', color=WONG['green'])
    axF.tick_params(axis='y', labelcolor=WONG['green'])
    axA.legend(lA + lF, [h.get_label() for h in lA + lF], fontsize=13, loc='best')

    # (b) force vs contact gap ---------------------------------------------
    axB.plot(gap_bb_f, Fsm, '-', color=WONG['vermillion'], lw=2.4, label='vs fuzzy edge (BB)')
    if gap_rg_f is not None:
        axB.plot(gap_rg_f, Fsm, color=WONG['blue'], lw=2.0, ls='--', label='vs body edge (Rg)')
    axB.axvline(0, color='k', ls=':', lw=1.5, alpha=0.7, label='edge contact (gap = 0)')
    axB.axhline(0, color='k', lw=0.8, alpha=0.3)
    if onset_i is not None:
        axB.axhline(thr, color='0.5', ls=':', lw=1.0, alpha=0.7)
        axB.plot(gap_bb_onset, Fsm[onset_i], 'o', color=WONG['vermillion'], ms=9, zorder=5)
        axB.annotate(f'force onset\n gap = {gap_bb_onset:.2f} $\\sigma$',
                     xy=(gap_bb_onset, Fsm[onset_i]),
                     xytext=(0.55, 0.75), textcoords='axes fraction', fontsize=13,
                     ha='left', arrowprops=dict(arrowstyle='->', color='0.4'))
    axB.invert_xaxis()          # piston descends (gap shrinks) => reads left->right
    axB.set_xlabel(r'contact gap  $z_\mathrm{piston}-z_\mathrm{top}$  ($\sigma$)')
    axB.set_ylabel(r'$F_z$  (LJ)'); axB.grid(alpha=0.3)
    axB.set_title('(b) force vs contact gap')
    axB.legend(fontsize=12, loc='upper left')

    out = PLOT_DIR / f'piston_gel_contact_{sim_name}{LEVEL_TAG}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()

    print(f'  fuzzy-edge (BB) force onset at gap = {gap_bb_onset:.2f} sigma '
          f'(threshold F_z > {thr:.1f} LJ)')
    if gap_rg_f is not None:
        print(f'  body-edge  (Rg) force onset at gap = {gap_rg_onset:.2f} sigma  '
              '(gap>0 => fuzzy layer thickness the piston crosses before touching the body)')


## 5 — Longitudinal modulus $M$: network stress at prescribed strain

Two independent estimates of the drained longitudinal modulus at the final
(relaxed) state, strain $\varepsilon = $ `comp_percent` ($\approx 10\%$):

- **Network:** $M_\mathrm{net} = \langle\sigma'_{zz}\rangle_\mathrm{membrane}/\varepsilon$
  — the network (effective) stress from §3 averaged over the membrane bins,
  divided by strain.  Error bar = 95 % CI from the bin-to-bin spatial scatter.
- **Piston:** $M_\mathrm{piston} = (F_z/A)/\varepsilon$ — the relaxed piston
  pressure divided by strain.  Error bar = piston-force fluctuation over the
  final averaging window propagated through $1/(A\varepsilon)$.

Agreement validates the Terzaghi network-stress decomposition. (Same comparison
as in `compression_analysis.ipynb`.)

In [ ]:
# ==========================================================================
#  5 — LONGITUDINAL MODULUS M:  network stress  at  prescribed strain
# ==========================================================================
# STRAIN-CONTROLLED: the piston is driven to a displacement = strain*L0_bb, so
# the APPLIED strain is comp_percent (= the detail level) -- this is the
# DENOMINATOR for M.  The network/piston STRESS is measured over the equilibrated
# plateau.  M = measured stress / applied strain.  The measured Rg and BB strains
# are reported as consistency checks (BB should ~= the applied strain).
eps_final = eps_applied
print(f'APPLIED strain (denominator)  eps = {eps_final:.4f}')
_bnd = f'; boundary/diagnostic {eps_boundary:.4f}' if HAS_BOUNDARY_STRAIN else ''
print(f'measured strain checks: Rg plateau {eps_measured:.4f}  BB plateau {eps_bb_measured:.4f}  '
      f'[end Rg {float(strain_eps[-1]):.4f}{_bnd}]')

# ---- Method 1: network (effective) stress  M = <sigma'_zz>_membrane / eps ----
# net_stack[-1] and in_mem come from section 3 (cell "3 - NETWORK ... STRESS").
Mnet_bins = net_stack[-1][in_mem] / eps_final          # per-bin M in the membrane
Mnet_bins = Mnet_bins[np.isfinite(Mnet_bins)]
M_net, M_net_lo, M_net_hi = mean_ci(Mnet_bins, ci_level)
ci_pct = int(ci_level * 100)
print(f'M_network = {M_net:.4f}  [{M_net_lo:.4f}, {M_net_hi:.4f}]  '
      f'({ci_pct}% CI, {len(Mnet_bins)} membrane bins)')

# ---- Method 2: piston pressure  M = (F_z/A) / eps ----
if has_piston_force:
    # Final plateau window = last coarse-stress epoch, matching the window that
    # produced the final network-stress snapshot.  Feed the RAW piston pressure
    # to a circular BLOCK BOOTSTRAP (accounts for autocorrelation of the undamped
    # piston directly); 95% CI = 2.5/97.5 percentiles of the bootstrap plateau
    # means, consistent with the network CI convention.
    # Equilibrated plateau = last `plateau_frac` of the production HOLD (piston
    # pressure has relaxed by then); more samples -> tighter, better-justified CI.
    _t0, _t1 = float(prod_ts[0]), float(prod_ts[-1])
    win_lo = _t1 - plateau_frac * (_t1 - _t0)
    _sel  = steps_pf >= win_lo
    P_win = P_piston_ts[_sel]
    if len(P_win) < 2:
        P_win = P_piston_ts[-10:]
    P_piston_final, P_lo, P_hi, _blk, _tau = block_bootstrap_ci(P_win, ci=ci_level)
    M_piston    = P_piston_final / eps_final
    M_piston_lo = P_lo / eps_final
    M_piston_hi = P_hi / eps_final
    M_piston_err = 0.5 * (M_piston_hi - M_piston_lo)
    print(f'M_piston  = {M_piston:.4f}  [{M_piston_lo:.4f}, {M_piston_hi:.4f}]  '
          f'({ci_pct}% block-bootstrap CI; block={_blk}, tau~{_tau:.1f}, '
          f'{len(P_win)} raw pts >= step {win_lo})')
    print(f'ratio M_piston / M_network = {M_piston / M_net:.4f}')

# ---- comparison plot with error bars ----
fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
ax.errorbar([0], [M_net], yerr=[[M_net - M_net_lo], [M_net_hi - M_net]],
            fmt='o', ms=13, color=WONG['blue'], capsize=8, lw=2.5,
            label=(f"network $\\sigma^{{\\prime}}_{{zz}}$   $M = {M_net:.3f}$\n"
                   f"{ci_pct}% CI [{M_net_lo:.3f}, {M_net_hi:.3f}]"))
if has_piston_force:
    ax.errorbar([1], [M_piston], yerr=[[M_piston - M_piston_lo], [M_piston_hi - M_piston]],
                fmt='s', ms=13, color=WONG['vermillion'], capsize=8, lw=2.5,
                label=(f"piston $F_z/A$   $M = {M_piston:.3f}$\n"
                       f"{ci_pct}% CI [{M_piston_lo:.3f}, {M_piston_hi:.3f}]"))
    ax.axhline(M_piston, color=WONG['vermillion'], ls='--', lw=1.2, alpha=0.5)
ax.axhline(M_net, color=WONG['blue'], ls='--', lw=1.2, alpha=0.5)
ax.set_xticks([0, 1])
ax.set_xticklabels([r"network ($\sigma'/\epsilon$)", r'piston ($P/\epsilon$)'], fontsize=17)
ax.set_ylabel(r'$M$  (LJ units)')
ax.set_title(f'Longitudinal modulus comparison\n{RUN_ID}  |  '
             f'$\\varepsilon_{{zz}} = {eps_final:.3f}$')
ax.set_xlim(-0.5, 1.5); ax.grid(axis='y', alpha=0.3); ax.legend(fontsize=14, loc='best')

out_M = PLOT_DIR / f'M_comparison_{sim_name}.png'
plt.savefig(out_M, dpi=150, bbox_inches='tight'); print('saved', out_M); plt.show()


## 6 — Stress-strain sweep summary: measured stress vs applied strain
One point per **applied-strain** level (`_c<level>`). The run is
**strain-controlled**: the piston is driven to displacement
$=\varepsilon\,L_0^{BB}$, so $\varepsilon$ on the x-axis is the *prescribed*
applied strain (`float(lvl)`), referenced to the seated bounding-box thickness.
The plotted $P=\langle F_z\rangle/A$ from the block-averaged piston force is the
**measured reaction** stress. The modulus is $M = P/\varepsilon$; its initial
secant slope $dP/d\varepsilon$ is the small-deformation longitudinal modulus.
The measured BB strain is reported alongside as a check (should $\approx$ the
applied strain). Set `COMP_LEVELS` to match `STRAIN_TARGETS` in
`triaxial_compression.batch`, and stage each level (sync cell) first.

In [ ]:
# Stress-strain sweep: measured piston stress vs APPLIED strain across levels.
# COMP_LEVELS (from Config) are the APPLIED STRAIN targets.  For each level we
# read the measured piston reaction pressure P = <F_z>/A over the last N_TAIL
# blocks of the hold; the modulus is M = P / applied_strain.  The measured BB
# strain is reported alongside as a check (should ~= the applied strain).
N_TAIL = 5   # trailing block rows averaged for the equilibrium (relaxed) value

eps_app_list, eps_bb_list, P_list = [], [], []
_L0bb_global = None   # seated BB thickness (level-1 drive start) = cumulative reference
for lvl in COMP_LEVELS:
    base = f'{sim_name}_c{lvl}'
    fF  = DATA_DIR / f'piston_force_avg_{base}.dat'
    fB  = DATA_DIR / f'box_dimensions_{base}.dat'
    fBB = DATA_DIR / f'gel_dimensions_bb_{base}.dat'
    if not (fF.exists() and fB.exists()):
        print(f'  level {lvl}: missing files, skipping')
        continue
    F, B = (np.atleast_2d(np.loadtxt(p, comments='#')) for p in (fF, fB))
    Fz = F[-N_TAIL:, 1].mean()
    A  = (B[-N_TAIL:, 1] * B[-N_TAIL:, 2]).mean()      # lx * ly
    P  = Fz / A
    eps_app = float(lvl)                               # APPLIED (prescribed) strain
    eps_bb  = np.nan
    if fBB.exists():
        BB = np.atleast_2d(np.loadtxt(fBB, comments='#'))
        if BB.size and BB.shape[1] >= 4 and BB[0, 3] != 0:
            if _L0bb_global is None:
                _L0bb_global = BB[0, 3]
            eps_bb = (_L0bb_global - BB[-N_TAIL:, 3].mean()) / _L0bb_global
    eps_app_list.append(eps_app); eps_bb_list.append(eps_bb); P_list.append(P)
    print(f'  level {lvl}: applied eps={eps_app:.4f}  measured BB eps={eps_bb:.4f}  '
          f'<F_z>={Fz:.4g}  A={A:.4g}  P={P:.4g}  M=P/eps={P/eps_app:.4g}')

eps_arr, epsbb_arr, P_arr = np.array(eps_app_list), np.array(eps_bb_list), np.array(P_list)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(eps_arr, P_arr, 'o-', lw=2, label='measured piston stress')
if len(eps_arr) >= 2:
    M_init = (P_arr[1] - P_arr[0]) / (eps_arr[1] - eps_arr[0])
    M_sec  = (P_arr[-1] - P_arr[0]) / (eps_arr[-1] - eps_arr[0])
    ax.set_title(f'Piston stress vs applied strain   |   $M_{{init}}\\approx${M_init:.3g}   $M_{{secant}}\\approx${M_sec:.3g}')
elif len(eps_arr) == 1:
    ax.set_title(f'Piston stress vs applied strain   |   $M = P/\\varepsilon \\approx${P_arr[0]/eps_arr[0]:.3g}')
ax.set_xlabel(r'applied strain  $\varepsilon$ (prescribed)')
ax.set_ylabel('piston stress  P = <F_z> / A  (LJ)')
ax.grid(alpha=0.3); ax.legend(fontsize=10)
out = PLOT_DIR / f'stress_strain_sweep_{sim_name}.png'
fig.tight_layout(); fig.savefig(out, dpi=150); print('saved', out); plt.show()


## 7 — Cooperative diffusivity $D_c$ (displacement Fourier fit), per strain level

Ported from `compression_analysis.ipynb` (Step 7, the **displacement** route).
Per-bin mean polymer $z$-displacement $u_z(z,t)$ is written by `fix avg_disp_z_poly`
in `triaxial_compression.lmp` (`disp_z_polymer_<sim_name>_c<level>.dat`), with the
displacement reference **reset at each level's hold onset** so $u_z(z,0)=0$ there.

Fit $D_c$ to the even-mode sine series (Dirichlet BCs — zero displacement at both
plates; zero IC):

$$u_z(z,t)=\sum_{k=1}^{N} A_k\left[1-e^{-4\pi^2 k^2\tau}\right]\sin\!\left(\frac{2\pi k z}{L_\mathrm{gel}}\right),\qquad \tau=\frac{D_c\,t}{L_\mathrm{gel}^2}.$$

For a fixed $D_c$ the amplitudes $\{A_k\}$ are the exact linear-least-squares
solution; $D_c$ is found by scalar minimisation of the total squared residual.
The sweep loops every level in `COMP_LEVELS` to give **one $D_c$ per applied strain**.

In [ ]:
# ==========================================================================
#  7 — Cooperative diffusivity D_c from polymer u_z(z,t): one D_c per strain
# ==========================================================================
# ---- D_c fit knobs ----
DC_N_MODES    = 5        # Fourier modes k = 1..N
DC_FRAC_EARLY = 1.0      # use the first this-fraction of hold snapshots (1.0 = all)
DC_TRIM_BINS  = 2        # interior gel bins dropped each side before fitting
DC_EDGE_MARGIN = 0       # extra gel-edge bins dropped when defining the domain
DC_BOUNDS     = (1e-6, 1.0)   # D_c search bounds (sigma^2 / tau)

def load_disp(lvl):
    """Read disp_z_polymer_<sim>_c<lvl>.dat -> dict(ts,z,Nc,uz) or None."""
    f = DATA_DIR / f'disp_z_polymer_{sim_name}_c{lvl}.dat'
    if not f.exists():
        return None
    snaps = read_ave_chunk_file(f)                 # cols: chunk, z, Ncount, mean_uz
    if not snaps:
        return None
    ts = np.array([s[0] for s in snaps], float)
    z  = snaps[0][1][:, 1]
    Nc = np.array([s[1][:, 2] for s in snaps])
    uz = np.array([s[1][:, 3] for s in snaps])
    return dict(ts=ts, z=z, Nc=Nc, uz=uz)

def fit_Dc(disp):
    """Sine-series fit of D_c to u_z(z,t).  Returns dict of fit + plotting arrays."""
    ts, z, Nc, uz = disp['ts'], disp['z'], disp['Nc'], disp['uz']
    gel = np.where(Nc[0] > Ncount_min)[0]
    if len(gel) < 4:
        return None
    iL = gel[0] + DC_EDGE_MARGIN; iR = gel[-1] - DC_EDGE_MARGIN + 1
    zg   = z[iL:iR]; uzg = uz[:, iL:iR]
    Lgel = (iR - iL) * binWidth
    zhat = (zg - zg[0]) / Lgel
    t_lj = (ts - ts[0]) * dt_lj
    early = np.where((t_lj > 0) & (t_lj <= DC_FRAC_EARLY * t_lj[-1]))[0]
    if len(early) < 2:
        return None
    mask = np.ones(len(zhat), bool); mask[:DC_TRIM_BINS] = False; mask[-DC_TRIM_BINS:] = False
    zf = zhat[mask]

    def basis(zh, Dc, t):
        tau = Dc * t / Lgel**2
        return np.column_stack([(1.0 - np.exp(-4.0*np.pi**2 * k**2 * tau)) * np.sin(2.0*np.pi*k*zh)
                                for k in range(1, DC_N_MODES + 1)])
    def amps(Dc):
        X = np.vstack([basis(zf, Dc, t_lj[i]) for i in early])
        y = np.concatenate([uzg[i][mask] for i in early])
        A, *_ = np.linalg.lstsq(X, y, rcond=None)
        return A
    def resid(Dc):
        A = amps(Dc)
        return float(sum(np.sum((basis(zf, Dc, t_lj[i]) @ A - uzg[i][mask])**2) for i in early))

    Dc = float(minimize_scalar(resid, bounds=DC_BOUNDS, method='bounded').x)
    A  = amps(Dc)
    y_all = np.concatenate([uzg[i][mask] for i in early])
    p_all = np.concatenate([basis(zf, Dc, t_lj[i]) @ A for i in early])
    ss_t  = np.sum((y_all - np.mean(y_all))**2)
    R2    = float(1.0 - np.sum((y_all - p_all)**2) / ss_t) if ss_t > 1e-30 else np.nan
    return dict(Dc=Dc, A=A, R2=R2, Lgel=Lgel, zhat=zhat, zf=zf, mask=mask,
                uzg=uzg, early=early, t_lj=t_lj, ts=ts, basis=basis, N=DC_N_MODES)

# ---- detailed fit + plot for the DETAIL level -----------------------------
_dd = load_disp(COMP_LEVEL)
_fd = fit_Dc(_dd) if _dd is not None else None
if _fd is None:
    print(f'D_c detail skipped for level _c{COMP_LEVEL}: '
          f'disp_z_polymer_{sim_name}_c{COMP_LEVEL}.dat missing or too few snapshots.')
else:
    zff  = np.linspace(0, 1, 400)
    uz_inf = sum(_fd['A'][k-1] * np.sin(2*np.pi*k*zff) for k in range(1, _fd['N']+1))
    fig, (axl, axr) = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)
    norm = Normalize(vmin=_fd['ts'][_fd['early'][0]], vmax=_fd['ts'][_fd['early'][-1]])
    cmap = plt.cm.viridis
    for i in _fd['early']:
        c = cmap(norm(_fd['ts'][i]))
        axl.plot(_fd['zhat'], _fd['uzg'][i], 'o-', color=c, ms=3, alpha=0.6)
        axr.plot(_fd['zf'], _fd['uzg'][i][_fd['mask']], 'o', color=c, ms=3, alpha=0.35)
        axr.plot(zff, _fd['basis'](zff, _fd['Dc'], _fd['t_lj'][i]) @ _fd['A'], '-', color=c, lw=2.0)
    axr.plot(zff, uz_inf, 'k--', lw=1.8, label=r'$u_z(t\to\infty)$')
    for ax in (axl, axr):
        ax.axhline(0, color='steelblue', ls=':', lw=1.5)
        ax.set(xlabel=r'$\hat{z}=z/L_\mathrm{gel}$', ylabel=r'$u_z$ ($\sigma$)', xlim=(0, 1))
        ax.grid(alpha=0.3)
    axl.set_title(r'Raw $u_z(z,t)$ — hold snapshots')
    axr.legend(fontsize=12)
    axr.set_title(rf"Sine fit ($N={_fd['N']}$): $D_c={_fd['Dc']:.2e}\ \sigma^2/\tau$, $R^2={_fd['R2']:.3f}$")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig.colorbar(sm, ax=[axl, axr], fraction=0.015, pad=0.04).set_label('Timestep')
    fig.suptitle(f'Cooperative diffusivity fit (level _c{COMP_LEVEL})  |  {sim_name}',
                 fontsize=12, fontweight='bold')
    out = PLOT_DIR / f'Dc_fourier_fit_{sim_name}_c{COMP_LEVEL}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
    print(f'  D_c(_c{COMP_LEVEL}) = {_fd["Dc"]:.4e} sigma^2/tau   R^2 = {_fd["R2"]:.3f}')

# ---- sweep: one D_c per applied strain level ------------------------------
dc_eps, dc_val, dc_r2 = [], [], []
for lvl in COMP_LEVELS:
    d = load_disp(lvl)
    fdc = fit_Dc(d) if d is not None else None
    if fdc is None:
        print(f'  level _c{lvl}: no D_c (missing disp file or too few snapshots)')
        continue
    dc_eps.append(float(lvl)); dc_val.append(fdc['Dc']); dc_r2.append(fdc['R2'])
    print(f'  level _c{lvl}: applied eps={float(lvl):.4f}  D_c={fdc["Dc"]:.4e}  R^2={fdc["R2"]:.3f}')

if dc_eps:
    _o = np.argsort(dc_eps)
    e = np.array(dc_eps)[_o]; v = np.array(dc_val)[_o]; r = np.array(dc_r2)[_o]
    fig, ax = plt.subplots(figsize=(7.5, 5.5), constrained_layout=True)
    ax.plot(e, v, 'o-', color=WONG['reddishpurple'], lw=2, ms=8)
    for xi, yi, ri in zip(e, v, r):
        ax.annotate(f'$R^2$={ri:.2f}', (xi, yi), textcoords='offset points',
                    xytext=(6, 6), fontsize=11, color='0.35')
    ax.set_xlabel(r'applied strain  $\varepsilon$ (prescribed)')
    ax.set_ylabel(r'$D_c$  ($\sigma^2/\tau$)')
    ax.set_title('Cooperative diffusivity vs applied strain')
    ax.grid(alpha=0.3)
    out = PLOT_DIR / f'Dc_vs_strain_{sim_name}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); print('saved', out); plt.show()
else:
    print('D_c sweep: no levels produced a fit (sync disp_z_polymer_*_c<level>.dat from the cluster).')
